# Install & Imports


In [ ]:

# Setup & Installation

# This section installs all required libraries and imports
# essential Python modules used throughout the notebook.
#
# Installed Packages:
# - google-generativeai: Access Google Generative AI models
# - langchain & langchain-google-genai: Build LLM workflows
# - networkx: Graph-based structures and algorithms
# - pandas: Data manipulation & tabular processing
# - pillow (PIL): Image processing
#
#
!pip -q install google-generativeai langchain langchain-google-genai networkx pandas pillow
!pip install gradio


# =========================================================
#  Imports
# =========================================================

import os, json, re
from typing import Any, Dict, List, Optional, Tuple, Set

from datetime import datetime
import pandas as pd

# If sendgrid not installed in your environment, run:
!pip install sendgrid
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# persisted log file (change path if you want it in /mnt/data)
EMAIL_LOG_FILE = "email_logs.json"

# Data & Graph Libraries
import networkx as nx

import pandas as pd

# Image handling
from PIL import Image

# Display outputs in Jupyter/Colab
from IPython.display import display

# Google Generative AI
import google.generativeai as genai

# LangChain Core Tools
from langchain.tools import BaseTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 13.0 MB/s eta 0:00:00


In [ ]:
# --- Step 2: Configure AWS Credentials ---
aws_access_key = "USE YOUR AWS ACCESS KEY"
aws_secret_key = "USE YOUR AWS SECRET KEY"
aws_region = "USE YOUR AWS REGION"   # replace with your region

!pip install boto3 folium

import boto3
import folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 5.4 MB/s eta 0:00:00


In [ ]:
# Lightweight Rich output (NO PANELS) — colored headings + pretty printed results
from rich.console import Console
from rich.pretty import Pretty
from typing import Any

console = Console()

def _print_heading(title: str, title_style: str):
    """Print a single-line colored heading (no box)."""
    console.print()  # blank line to separate blocks
    console.print(f"[{title_style}]{title}[/{title_style}]")

def _print_content(content: Any, content_style: str = "bright_white"):
    """Print content: dicts/lists use Pretty, else plain text (with style)."""
    if isinstance(content, (dict, list)):
        # Pretty will still color keys/values nicely in dark mode
        console.print(Pretty(content), style=content_style)
    else:
        console.print(str(content), style=content_style)

# === Keep the same function names used across your notebook ===
# Colors chosen to match your previous professional palette:
# - Agent Response -> yellow
# - Tool Call      -> cyan
# - Tool Result    -> green
# - Thinking       -> bright_white
# - Warning/Error  -> red

def show_agent_response(content: Any):
    _print_heading("GRABNAVI RESPONSE", "bold black")
    _print_content(content, content_style="black")

def show_tool_call(tool_name: str, args: Any):
    _print_heading(f"TOOL CALL: {tool_name}", "bold black")
    # print args clearly
    _print_content({"args": args}, content_style="black")

def show_tool_result(tool_name: str, result: Any):
    _print_heading(f"TOOL RESULT: {tool_name}", "bold green")
    _print_content(result, content_style="black")

def show_thinking(title: str, content: Any):
    _print_heading(f"{title}", "bold black")
    _print_content(content, content_style="black")

def show_warning(content: Any):
    _print_heading("WARNING", "bold red")
    _print_content(content, content_style="black")


# API Key + Base Graph Setup


In [ ]:
# This section:
# 1. Configures the Google Gemini API key for LLM access.
# 2. Creates a base city road network (graph) where:
#    - Nodes represent locations
#    - Edges represent roads
#    - Weights represent approximate travel time (in minutes)
# 3. Defines a global path for evidence images (can be changed later).
# =========================================================

# ---------------------------------------------------------
# 1. Configure Google Generative AI API
# ---------------------------------------------------------
os.environ["GOOGLE_API_KEY"] = "USE YOUR GOOGLE API KEY"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# ---------------------------------------------------------
# 2. Define City Road Network (Graph)
# ---------------------------------------------------------
CITY = nx.Graph()

# Add nodes + edges with weights (~ travel time in minutes)
CITY.add_edge("Restaurant", "MainRoad", weight=6)
CITY.add_edge("MainRoad", "Airport", weight=20)
CITY.add_edge("MainRoad", "MG_Bypass", weight=8)
CITY.add_edge("MG_Bypass", "Airport", weight=12)
CITY.add_edge("MainRoad", "Downtown", weight=10)
CITY.add_edge("Downtown", "Airport", weight=14)

# ---------------------------------------------------------
# 3. Global Evidence Image Path
# ---------------------------------------------------------
EVIDENCE_IMAGE = "/content/download1.jpg"


# VisionService (Gemini Captioning + Decision, Shared)

In [ ]:

# This class wraps Gemini models (image + text) into a simple
# vision analysis service for package damage detection.
#
# Features:
#   - caption_image()   → Generates descriptive caption from an image
#   - decide_damage()   → Decides (via Gemini) if packaging is damaged
#   - analyze_photos()  → End-to-end process {captions, decision}
#
# Notes:
#   - Uses a heuristic for compromised packaging (open/torn/exposed).
#   - Provides a safe fallback if LLM JSON parsing fails.
#   - Can be accessed as a singleton via get_vision().
# =========================================================
from pathlib import Path

class VisionService:
    """
    Centralized vision utility using Gemini:
      - caption_image(image_path)  → caption via Gemini (image input)
      - decide_damage(caption)     → JSON decision via Gemini (text input)
      - analyze_photos(cust, drv)  → end-to-end {captions, decision}

    Includes a heuristic for compromised packaging:
    (open, torn, exposed contents, etc.).
    """

    # -----------------------------------------------------
    #  Initialization
    # -----------------------------------------------------
    def __init__(self,
                 image_model: str = "gemini-1.5-flash",
                 text_model: str  = "gemini-1.5-flash"):
         # Configure Gemini models (separate for vision & text)
        self.image_model_name = image_model
        self.text_model_name  = text_model
        self.image_model = genai.GenerativeModel(self.image_model_name)
        self.text_model  = genai.GenerativeModel(self.text_model_name)


    # ---------- 1. Caption an Image ----------
    def caption_image(self, image_path: str) -> str:
        """Generates a descriptive caption for a given image file."""
        p = Path(image_path)
        if not p.exists():
            return "(no image found)"
        try:
            img = Image.open(image_path).convert("RGB")
            prompt = "Describe clearly what you see. Focus on the condition of the package/box/bag."
            resp = self.image_model.generate_content([prompt, img])
            return (resp.text or "").strip()
        except Exception as e:
            return f"(caption_error: {e})"

# ---------- 2. Heuristic: Compromised Packaging ----------
    @staticmethod
    def _compromised_from_caption(txt: str) -> bool:
        """Check if caption suggests packaging compromise (open/torn/exposed)."""
        low = txt.lower()
        keys = [
            "open", "opened", "torn", "rip", "ripped", "exposed",
            "contents sticking out", "contents showing", "broken seal",
            "bag sticking out", "hole", "split seam"
        ]
        return any(k in low for k in keys)

    # ----------3. decision ----------
    def decide_damage(self, caption_text: str) -> Dict[str, Any]:
        """
        Uses Gemini (text model) to classify package damage.
        Returns JSON with:
          {is_damaged: bool, confidence: float, message_to_customer: str, reason: str}
        Falls back to heuristic if LLM parsing fails.
        """
        schema = """
Return ONLY valid JSON:
{"is_damaged": true|false, "confidence": number, "message_to_customer": string, "reason": string}
"""
        fewshot = """
Caption: "gray poly mailer with contents sticking out of the open seam"
JSON: {"is_damaged": true, "confidence": 0.78, "message_to_customer": "We detected compromised packaging. We'll make it right.", "reason": "packaging compromised"}

Caption: "sealed shipping bag on a table"
JSON: {"is_damaged": false, "confidence": 0.7, "message_to_customer": "No visible damage detected from the photo.", "reason": "packaging sealed"}
"""
        prompt = (
            "You are a delivery dispute resolver. Given an image caption, decide if the package is damaged.\n"
            "- Consider packaging compromised (open/torn/exposed) as damaged even if contents look intact.\n"
            "- Consider crushed/wet/leaking as damaged.\n"
            "- Consider sealed/undamaged as not damaged.\n\n"
            f"{schema}\n{fewshot}\nCaption:\n{caption_text}\n\nJSON:"
        )
        try:
            resp = self.text_model.generate_content(prompt)
            text = (resp.text or "").strip()
            data = json.loads(text)
            if {"is_damaged","confidence","message_to_customer","reason"} <= set(data.keys()):
                c = float(data.get("confidence", 0.0))
                data["confidence"] = max(0.0, min(1.0, c))
                return data
        except Exception:
            pass

      # ----- Fallback Heuristic -----
        damaged_kw = self._compromised_from_caption(caption_text) or any(
            k in caption_text.lower() for k in
            ["crushed","torn","leak","leaking","spilled","wet","broken","dented","damaged"]
        )
        msg = ("We detected compromised packaging. We'll make it right."
               if damaged_kw else "We inspected the photo and found no visible damage.")
        return {
            "is_damaged": damaged_kw,
            "confidence": 0.68 if damaged_kw else 0.55,
            "message_to_customer": msg,
            "reason": "heuristic"
        }
 # ---------- 4. End-to-End: Analyze Two Photos ----------
    def analyze_photos(self, customer_photo: str, driver_photo: str) -> Dict[str, Any]:
        """
        Runs full pipeline:
        1. Caption both customer + driver photos.
        2. Make a combined decision.
        3. Override with heuristic if needed.
        """
        cap_c = self.caption_image(customer_photo)
        cap_d = self.caption_image(driver_photo)
        combined = f"customer: {cap_c} | driver: {cap_d}"
        decision = self.decide_damage(combined)

        # Explicit compromised-packaging override
        if self._compromised_from_caption(combined) and not decision.get("is_damaged", False):
            decision["is_damaged"] = True
            decision["confidence"] = max(decision.get("confidence", 0.0), 0.7)
            decision["reason"]     = (decision.get("reason","") + "+compromised_packaging").strip("+")

        return {
            "captions": {"customer": cap_c, "driver": cap_d, "combined": combined},
            "decision": decision
        }

# =========================================================
# Singleton Accessor
# =========================================================
_VISION = VisionService()
def get_vision() -> VisionService:
  """Access the shared VisionService instance."""
  return _VISION

# Tool Implementations (use shared VisionService)

In [ ]:

# These functions act as tools accessible by the LangChain Agent.
# Each tool simulates a real-world system call:
#   1. Traffic info
#   2. Route calculation
#   3. Notifications
#   4. Evidence collection & analysis
#   5. Refund / Driver exoneration
#   6. Restaurant overload handling
#
# Notes:
#   - Evidence analysis uses VisionService (Gemini + heuristic).
#   - Graph operations (ETA/paths) are handled via NetworkX.
# =========================================================

# ---------------------------------------------------------
# 1. Traffic Check
# ---------------------------------------------------------
def tool_check_traffic(location: str):
    """
    Simulates checking for traffic incidents at a location.
    Returns an incident type and possible delay.
    """
    return {
        "location": location,
        "incident": "accident" if location == "MainRoad" else "clear",
        "delay_minutes": 18 if location == "MainRoad" else 0
    }

# ---------------------------------------------------------
# 2️. Route Calculation
# ---------------------------------------------------------

def tool_calculate_alternative_route(start: str, end: str):
   """
    Calculates shortest path between two nodes in CITY graph.
    Returns path + ETA (weight sum).
    """
   try:
        path = nx.shortest_path(CITY, start, end, weight="weight")
        eta  = nx.path_weight(CITY, path, weight="weight")
   except Exception:
        path, eta = [start, end], 999
   return {"start": start, "end": end, "path": path, "eta_minutes": int(eta)}

# ---------------------------------------------------------
# 3️. User Notification
# ---------------------------------------------------------
def tool_notify_user(role: str, message: str):
    return {"role": role, "message": message}

# ---------------------------------------------------------
# 4️. Collect Evidence
# ---------------------------------------------------------
def tool_collect_evidence(order_id: str):
    # In Colab, set EVIDENCE_IMAGE via the upload cell.
    return {"order_id": order_id,
            "customer_photo": EVIDENCE_IMAGE,
            "driver_photo":   EVIDENCE_IMAGE}

# ---------------------------------------------------------
# 5️. Analyze Evidence
# ---------------------------------------------------------
def tool_analyze_evidence(customer_photo: str, driver_photo: str):
    res = get_vision().analyze_photos(customer_photo, driver_photo)
    dec = res["decision"]
    return {
        "damage_detected": bool(dec.get("is_damaged", False)),
        "confidence": float(dec.get("confidence", 0.0)),
        "message_to_customer": dec.get("message_to_customer", ""),
        "reason": dec.get("reason", ""),
        "captions": res["captions"]
    }


# ---------------------------------------------------------
# 6️. Issue Refund
# ---------------------------------------------------------
def tool_issue_refund(order_id: str, amount: float):
    return {"order_id": order_id, "refunded": True, "amount": amount}


# ---------------------------------------------------------
# 7️. Exonerate Driver
# ---------------------------------------------------------
def tool_exonerate_driver(order_id: str):
    return {"order_id": order_id, "driver_fault": False}

# ---------------------------------------------------------
# 8️. GrabFOOD: Restaurant Overload Handling
# ---------------------------------------------------------
def tool_get_merchant_status(merchant_id: str) -> Dict[str, Any]:
    """
    Simulate merchant status. Returns prep time in minutes.
    """
    # For demo, force an overloaded case (40 min)
    return {"merchant_id": merchant_id, "prep_time": 40}

def tool_notify_customer(message: str, voucher: bool = False) -> Dict[str, Any]:
    payload = {"role": "customer", "message": message}
    if voucher:
        payload["voucher_issued"] = True
        payload["voucher_code"] = "DELAY-5"  # demo
    return payload

def tool_re_route_driver(driver_id: str, task: str) -> Dict[str, Any]:
    return {"driver_id": driver_id, "assigned_task": task}

def tool_get_nearby_merchants(cuisine: str, max_wait: int = 20) -> List[str]:
    # Demo suggestions
    return ["FastBites Diner", "QuickEats Express", f"{cuisine} Hub (ETA 15m)"]

# ---------------------------------------------------------
# 9️. Customer Complaint Handling
# ---------------------------------------------------------
def analyze_sentiment(text: str) -> str:
    """Rule-based sentiment detector for demo."""
    text = text.lower()
    if any(word in text for word in ["angry", "frustrated", "worst", "hate", "late", "not coming", "??", "!!"]):
        return "negative"
    elif any(word in text for word in ["thanks", "ok", "cool", "great", "happy"]):
        return "positive"
    else:
        return "neutral"


def handle_customer_delay_complaint(customer_message: str):
    print("Customer Complaint:", customer_message)

    # Step 0: Sentiment Analysis
    sentiment = analyze_sentiment(customer_message)

    if sentiment == "negative":
        print("Escalation detected: Customer is frustrated.")
        notify_customer(
            "I'm really sorry your food is delayed. I understand how frustrating this can be. "
            "I've issued you a voucher for the inconvenience, and I'm connecting you to a support agent right away."
        )
        escalate_to_human_agent()
    else:
        # Normal reassurance flow
        notify_customer(
            "Your order is on the way and may take a little longer than expected. "
            "Thanks for your patience!"
        )


def notify_customer(message: str):
    print("Notify Customer:", message)


def escalate_to_human_agent(problem=None, context=None):
    print(f"Escalating to human agent. Problem: {problem}, Context: {context}")
    return {"status": "escalated", "problem": problem}


# System Prompt, Text LLM, Incident Detector (Gemini)

In [ ]:
# This section sets up:
#   1. SYSTEM_PROMPT → Governs agent reasoning & tool usage rules.
#   2. DET_PROMPT    → Schema for detecting incidents from user text.
#   3. llm_text()    → Factory for Gemini text model via LangChain.
#   4. detect_incidents_llm() → Detects traffic/damage incidents
#                               (LLM-first, with heuristic fallback).
# =========================================================

# ---------- 1. System Prompt ----------
SYSTEM_PROMPT = """
You are Project Synapse, an autonomous last-mile coordinator.
Rules:
- Traffic: use check_traffic -> calculate_alternative_route -> notify_user (role='customer').
- Damaged Package: collect_evidence -> analyze_evidence;
  if damage_detected and confidence>=0.58 then issue_instant_refund(amount=10.0) + exonerate_driver;
  finally notify_user (role='customer').
- Overloaded Restaurant (GrabFOOD): get_merchant_status -> notify_customer(+voucher) -> re_route_driver -> optionally get_nearby_merchants for alternatives.
- Be concise and resolve in minimum steps. Always use tools to act; do not explain.
"""

# ---------- 2. Detection Schema Prompt ----------
DET_PROMPT = """You classify delivery/driver support messages.
Return ONLY valid JSON, no prose.

Labels: ["Traffic Obstruction","Damaged Package","Overloaded Restaurant"].

JSON:
{
  "incidents": [
    {
      "label": "Traffic Obstruction" | "Damaged Package" | "Overloaded Restaurant",
      "fields": {
        "location": "<string|null>",
        "start": "<string|null>",
        "end": "<string|null>",
        "order_id": "<string|null>",
        "merchant_id": "<string|null>",
        "driver_id": "<string|null>",
        "cuisine": "<string|null>",
        "prep_time": "<number|null>"
      }
    }
  ]
}

Guidelines:
- 'jam/accident/blockage/delay/congestion' => Traffic Obstruction.
- 'damaged/spilled/wet/broken/torn/leak/open/exposed' => Damaged Package.
- 'overloaded/long prep/40-minute/kitchen prep time/long wait' => Overloaded Restaurant.
- Emit multiple incidents if both appear.
- Fill fields if tokens present (e.g., 'MainRoad','Restaurant','Airport','ORD123','merchant/drv ids').
"""
# ---------- 3. Text Model Factory ----------
def llm_text() -> ChatGoogleGenerativeAI:
    """Gemini-1.5-flash wrapped for LangChain."""
    return ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=os.environ.get("GOOGLE_API_KEY",""),
        temperature=0.0
    )

# ---------- 4. Incident Detection ----------

def detect_incidents_llm(user_input: str) -> List[Dict[str, Any]]:
    """
    Detects incidents from free-form user text.
    """
    prompt = f"{DET_PROMPT}\n\nText:\n{user_input}\n\nJSON:"
    model = genai.GenerativeModel("gemini-1.5-flash")

    # --- Primary: Gemini JSON ---
    try:
        resp = model.generate_content(prompt)
        data = json.loads((resp.text or "").strip())
        incidents = data.get("incidents", [])
        valid_labels = {"Traffic Obstruction","Damaged Package","Overloaded Restaurant"}
        valid = [i for i in incidents if i.get("label") in valid_labels]
        for i in valid:
            i.setdefault("fields", {})
            f = i["fields"]
            # ensure keys exist
            for k in ["location","start","end","order_id","merchant_id","driver_id","cuisine","prep_time"]:
                f.setdefault(k, None)
        return valid
    except Exception:
        pass

    # --- Fallback: Heuristic Rules (simple, not optimized) ---
    out: List[Dict[str, Any]] = []
    low = user_input.lower()

    # Traffic detection
    if any(k in low for k in ["traffic","jam","blocked","blockage","accident","congestion","delay"]):
        out.append({"label": "Traffic Obstruction",
                    "fields": {"location": "MainRoad" if "mainroad" in low else None,
                               "start": "Restaurant" if "restaurant" in low else None,
                               "end": "Airport" if "airport" in low else None,
                               "order_id": None,
                               "merchant_id": None, "driver_id": None, "cuisine": None, "prep_time": None}})
    # Damage package detection
    if any(k in low for k in ["damage","damaged","broken","spilled","wet","torn","leak","open","exposed"]):
        m = re.search(r"\bORD\w+\b", user_input.upper())
        oid = m.group(0) if m else None
        out.append({"label": "Damaged Package",
                    "fields": {"location": None,"start": None,"end": None,"order_id": oid,
                               "merchant_id": None,"driver_id": None,"cuisine": None,"prep_time": None}})
    # Overloaded Restaurant detection
    if any(k in low for k in ["overloaded","long prep","kitchen prep","prep time","long wait","40-minute","40 minute","40min"]):
        # naive pulls
        m_id = None
        m = re.search(r"merchant[:\s]+([A-Za-z0-9_-]+)", user_input)
        if m: m_id = m.group(1)
        driver_id = None
        m2 = re.search(r"driver[:\s]+([A-Za-z0-9_-]+)", user_input)
        if m2: driver_id = m2.group(1)
        cuisine = "Food"
        out.append({"label": "Overloaded Restaurant",
                    "fields": {"location": None,"start": None,"end": None,"order_id": None,
                               "merchant_id": m_id or "MerchantX", "driver_id": driver_id or "DRV101",
                               "cuisine": cuisine, "prep_time": 40}})
    return out

In [ ]:

# This cell defines:
# 1. LangChain-style **tool wrappers** (converting helper
#    functions into callable LangChain tools).
# 2. An **agent factory** to build a tool-calling LLM agent.
# 3. A **logger utility** to capture agent steps/actions.
# =========================================================

# -------------------------------
#  Traffic & Route Management
# -------------------------------
class CheckTrafficTool(BaseTool):
    name: str = "check_traffic"
    description: str = "Check traffic at location. Input: location"
    def _run(self, location: str): return json.dumps(tool_check_traffic(location))
    async def _arun(self, location: str): return self._run(location)

class CalcAltRouteTool(BaseTool):
    name: str = "calculate_alternative_route"
    description: str = "Get alternative route. Input: start, end"
    def _run(self, start: str, end: str): return json.dumps(tool_calculate_alternative_route(start, end))
    async def _arun(self, start: str, end: str): return self._run(start, end)

# -------------------------------
#  Notifications
# -------------------------------
class NotifyUserTool(BaseTool):
    name: str = "notify_user"
    description: str = "Notify user. Input: role, message"
    def _run(self, role: str, message: str): return json.dumps(tool_notify_user(role, message))
    async def _arun(self, role: str, message: str): return self._run(role, message)

# -------------------------------
#  Evidence Handling
# -------------------------------
class CollectEvidenceTool(BaseTool):
    name: str = "collect_evidence"
    description: str = "Collect evidence photos. Input: order_id"
    def _run(self, order_id: str): return json.dumps(tool_collect_evidence(order_id))
    async def _arun(self, order_id: str): return self._run(order_id)

class AnalyzeEvidenceTool(BaseTool):
    name: str = "analyze_evidence"
    description: str = "Analyze evidence. Input: customer_photo, driver_photo"
    def _run(self, customer_photo: str, driver_photo: str): return json.dumps(tool_analyze_evidence(customer_photo, driver_photo))
    async def _arun(self, customer_photo: str, driver_photo: str): return self._run(customer_photo, driver_photo)


# -------------------------------
#  Order Resolution
# -------------------------------
class IssueRefundTool(BaseTool):
    name: str = "issue_instant_refund"
    description: str = "Refund customer. Input: order_id, amount"
    def _run(self, order_id: str, amount: float): return json.dumps(tool_issue_refund(order_id, amount))
    async def _arun(self, order_id: str, amount: float): return self._run(order_id, amount)

class ExonerateDriverTool(BaseTool):
    name: str = "exonerate_driver"
    description: str = "Exonerate driver. Input: order_id"
    def _run(self, order_id: str): return json.dumps(tool_exonerate_driver(order_id))
    async def _arun(self, order_id: str): return self._run(order_id)

# -------------------------------
#  GrabFOOD Restaurant Ops
# -------------------------------
class GetMerchantStatusTool(BaseTool):
    name: str = "get_merchant_status"
    description: str = "Get merchant kitchen status. Input: merchant_id"
    def _run(self, merchant_id: str): return json.dumps(tool_get_merchant_status(merchant_id))
    async def _arun(self, merchant_id: str): return self._run(merchant_id)

class NotifyCustomerTool(BaseTool):
    name: str = "notify_customer"
    description: str = "Notify customer with optional voucher. Input: message, voucher(bool)"
    def _run(self, message: str, voucher: bool=False): return json.dumps(tool_notify_customer(message, voucher))
    async def _arun(self, message: str, voucher: bool=False): return self._run(message, voucher)

class ReRouteDriverTool(BaseTool):
    name: str = "re_route_driver"
    description: str = "Re-route driver to a short task. Input: driver_id, task"
    def _run(self, driver_id: str, task: str): return json.dumps(tool_re_route_driver(driver_id, task))
    async def _arun(self, driver_id: str, task: str): return self._run(driver_id, task)

class GetNearbyMerchantsTool(BaseTool):
    name: str = "get_nearby_merchants"
    description: str = "Find alternatives. Input: cuisine, max_wait"
    def _run(self, cuisine: str, max_wait: int=20): return json.dumps(tool_get_nearby_merchants(cuisine, max_wait))
    async def _arun(self, cuisine: str, max_wait: int=20): return self._run(cuisine, max_wait)


# =========================================================
#  Agent Factory
# =========================================================
def make_agent(return_steps: bool = True) -> AgentExecutor:
    """
    Factory function to create a LangChain agent
    with access to all registered tools.
    """
    tools = [
        # existing
        CheckTrafficTool(),
        CalcAltRouteTool(),
        NotifyUserTool(),
        CollectEvidenceTool(),
        AnalyzeEvidenceTool(),
        IssueRefundTool(),
        ExonerateDriverTool(),
        # new for GrabFOOD
        GetMerchantStatusTool(),
        NotifyCustomerTool(),
        ReRouteDriverTool(),
        GetNearbyMerchantsTool(),
    ]

    # Prompt template: system role + human input + scratchpad
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    # Build a tool-calling agent
    agent = create_tool_calling_agent(llm_text(), tools, prompt)

    # Return agent executor (with option to log intermediate steps)
    return AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=return_steps)

# =========================================================
#  Logger Utility
# =========================================================
def extract_actions(intermediate_steps, incident_label: str) -> List[Dict[str, Any]]:
    """
    Extracts agent actions into a structured log format.

    Args:
        intermediate_steps: List of (action, observation) tuples
        incident_label: Label for the current incident

    Returns:
        List of structured action dictionaries with timestamps
    """
    actions: List[Dict[str, Any]] = []
    from datetime import datetime
    for i, (agent_action, observation) in enumerate(intermediate_steps, start=1):
        tool_name = getattr(agent_action, "tool", "unknown")
        tool_input = getattr(agent_action, "tool_input", {})
        actions.append({
            "step": i,
            "incident": incident_label,
            "tool": tool_name,
            "args": tool_input,
            "observation": observation,
            "ts": datetime.utcnow().isoformat() + "Z"
        })
    return actions

In [ ]:
# @title Single-incident handler (Overloaded Restaurant)
# 📌 This cell defines the handler function for restaurant overloads:
#
# handle_overloaded_restaurant_case(...)
#   - Triggered when a restaurant is experiencing high load (long prep times).
#   - Flow:
#       1. Get merchant status (prep time).
#       2. If prep ≥ 30 minutes:
#            → Notify customer + issue small voucher
#            → Re-route driver to a nearby short task
#            → Suggest alternative nearby merchants
#       3. Else (prep < 30 minutes):
#            → Just notify customer with current prep time
#   - Every step is logged with timestamps for traceability.
#
# Returns:
#   {
#       "label": "Overloaded Restaurant",
#       "context": { merchant_id, driver_id, cuisine },
#       "steps": [ ... ],
#       "resolution": "summary string"
#   }


from datetime import datetime

def handle_traffic_case(agent: AgentExecutor, user_input: str, fields: Dict[str, Optional[str]]) -> Dict[str, Any]:
    location = fields.get("location") or ("MainRoad" if "MainRoad" in user_input else "MainRoad")
    start    = fields.get("start")    or ("Restaurant" if "Restaurant" in user_input else "Restaurant")
    end      = fields.get("end")      or ("Airport" if "Airport" in user_input else "Airport")

    instruction = (
        f"Traffic incident at {location}. Start={start} End={end}. "
        f"Use: check_traffic -> calculate_alternative_route -> notify_user(role='customer')."
    )
    result = agent.invoke({"input": instruction})
    actions = extract_actions(result.get("intermediate_steps", []), "Traffic Obstruction")
    return {
        "label": "Traffic Obstruction",
        "context": {"location": location, "start": start, "end": end},
        "steps": actions,
        "resolution": f"Traffic Obstruction handled in {len(actions)} steps"
    }

def handle_damage_case(agent: AgentExecutor, user_input: str, fields: Dict[str, Optional[str]]) -> Dict[str, Any]:
    oid = fields.get("order_id")
    if not oid:
        m = re.search(r"\bORD\w+\b", user_input.upper())
        oid = m.group(0) if m else "UNKNOWN"

    steps: List[Dict[str, Any]] = []

    ev = tool_collect_evidence(oid)
    steps.append({
        "step": len(steps)+1, "incident": "Damaged Package", "tool": "collect_evidence",
        "args": {"order_id": oid}, "observation": ev, "ts": datetime.utcnow().isoformat() + "Z"
    })

    ana = tool_analyze_evidence(ev["customer_photo"], ev["driver_photo"])
    steps.append({
        "step": len(steps)+1, "incident": "Damaged Package", "tool": "analyze_evidence",
        "args": {"customer_photo": "customer_photo", "driver_photo": "driver_photo"},
        "observation": ana, "ts": datetime.utcnow().isoformat() + "Z"
    })

    refund_threshold = 0.58
    message = ana.get("message_to_customer") or "Your claim has been reviewed."

    if ana.get("damage_detected") and float(ana.get("confidence", 0)) >= refund_threshold:
        r = tool_issue_refund(oid, amount=10.0)
        steps.append({
            "step": len(steps)+1, "incident": "Damaged Package", "tool": "issue_instant_refund",
            "args": {"order_id": oid, "amount": 10.0}, "observation": r, "ts": datetime.utcnow().isoformat() + "Z"
        })
        ex = tool_exonerate_driver(oid)
        steps.append({
            "step": len(steps)+1, "incident": "Damaged Package", "tool": "exonerate_driver",
            "args": {"order_id": oid}, "observation": ex, "ts": datetime.utcnow().isoformat() + "Z"
        })
        if "refund" not in message.lower():
            message = "We detected damage in your photos and issued an instant refund. The driver has been cleared."
    else:
        if "no damage" not in message.lower():
            message = "We reviewed the photos and did not detect visible damage."

    note = tool_notify_user(role="customer", message=message)
    steps.append({
        "step": len(steps)+1, "incident": "Damaged Package", "tool": "notify_user",
        "args": {"role": "customer", "message": message}, "observation": note, "ts": datetime.utcnow().isoformat() + "Z"
    })

    return {
        "label": "Damaged Package",
        "context": {"order_id": oid},
        "steps": steps,
        "resolution": f"Damaged Package handled in {len(steps)} steps"
    }

# --- NEW: GrabFOOD Overloaded Restaurant handler ---
def handle_overloaded_restaurant_case(
    agent: AgentExecutor,
    fields: Dict[str, Optional[str]]
) -> Dict[str, Any]:
    merchant_id = fields.get("merchant_id") or "MerchantX"
    driver_id   = fields.get("driver_id") or "DRV101"
    cuisine     = fields.get("cuisine") or "Food"
    prep_time   = fields.get("prep_time") or 40

    steps: List[Dict[str, Any]] = []

    # Step 1: get_merchant_status
    status = tool_get_merchant_status(merchant_id)
    steps.append({
        "step": len(steps)+1, "incident": "Overloaded Restaurant", "tool": "get_merchant_status",
        "args": {"merchant_id": merchant_id}, "observation": status, "ts": datetime.utcnow().isoformat()+"Z"
    })

  # Overloaded condition → prep time ≥ 30 mins
    if int(status.get("prep_time", prep_time)) >= 30:
        # Step 2: notify_customer + voucher
        msg = f"Your order from {merchant_id} has a longer prep time (~{status.get('prep_time', prep_time)} min). We’ve issued a small voucher for the delay."
        note = tool_notify_customer(msg, voucher=True)
        steps.append({
            "step": len(steps)+1, "incident": "Overloaded Restaurant", "tool": "notify_customer",
            "args": {"message": msg, "voucher": True}, "observation": note, "ts": datetime.utcnow().isoformat()+"Z"
        })

        # Step 3: re_route_driver to a nearby short task
        rr = tool_re_route_driver(driver_id, "short nearby delivery")
        steps.append({
            "step": len(steps)+1, "incident": "Overloaded Restaurant", "tool": "re_route_driver",
            "args": {"driver_id": driver_id, "task": "short nearby delivery"}, "observation": rr, "ts": datetime.utcnow().isoformat()+"Z"
        })

        # Step 4: optionally suggest alternatives
        alts = tool_get_nearby_merchants(cuisine, max_wait=20)
        steps.append({
            "step": len(steps)+1, "incident": "Overloaded Restaurant", "tool": "get_nearby_merchants",
            "args": {"cuisine": cuisine, "max_wait": 20}, "observation": alts, "ts": datetime.utcnow().isoformat()+"Z"
        })
        resolution = "Overloaded Restaurant handled with notify+voucher, driver rerouted, alternatives suggested."
    else:
      # Non-overloaded → just notify customer
        msg = f"Current prep time at {merchant_id} is {status.get('prep_time', prep_time)} minutes."
        note = tool_notify_customer(msg, voucher=False)
        steps.append({
            "step": len(steps)+1, "incident": "Overloaded Restaurant", "tool": "notify_customer",
            "args": {"message": msg, "voucher": False}, "observation": note, "ts": datetime.utcnow().isoformat()+"Z"
        })
        resolution = "Merchant prep time acceptable; customer informed."

    return {
        "label": "Overloaded Restaurant",
        "context": {"merchant_id": merchant_id, "driver_id": driver_id, "cuisine": cuisine},
        "steps": steps,
        "resolution": resolution
    }


# Compound Problem Handling

In [ ]:
# === Compound problem priority module (replaces old Compound Problem Handling) ===
from typing import List, Dict, Any, Tuple
import asyncio
import json
import time

# 1) Priority map: lower number = higher priority
PRIORITY_MAP = {
    "safety_incident": 1,        # driver injured, assault, theft
    "damaged_item": 2,           # broken/contaminated food
    "overloaded_restaurant": 3,  # merchant delays > SLA
    "traffic_obstruction": 4,    # jam/accident affecting ETA
    "missing_driver": 5,         # driver offline/missing
    "payment_issue": 6,          # charge failed / refund required
    "customer_complaint": 7,     # lower severity feedback
    "other": 99
}

# 2) Keyword hints to map text -> problems (expandable)
PROBLEM_KEYWORDS = {
    "damaged_item": ["damaged", "smashed", "broken", "leaking", "spoiled", "stale", "cold"],
    "overloaded_restaurant": ["delay", "long prep", "40-minute", "busy kitchen", "overloaded", "backlog"],
    "traffic_obstruction": ["traffic", "jam", "stuck", "road block", "congestion", "accident"],
    "missing_driver": ["driver missing", "driver offline", "driver not responding", "lost", "no-show"],
    "payment_issue": ["payment failed", "charge", "refund", "billing", "paid but not delivered"],
    "safety_incident": ["injured", "hurt", "assault", "robbed", "danger", "threat"],
    "customer_complaint": ["not coming on time", "worst", "late", "bad service", "cold food"],
}

import json
from typing import List, Tuple, Dict, Any

# stricter PROBLEM_KEYWORDS: prefer phrases and whole words
# ===== Improved strict problem-keywords and normalization =====
PROBLEM_KEYWORDS_STRICT = {
    "damaged_item": [r"\bdamaged\b", r"\bspoiled\b", r"\bsmashed\b", r"\bbroken\b", r"\bbag open\b", r"\bcontents exposed\b", r"\bcold food\b"],
    "overloaded_restaurant": [r"\b(long prep|prep time|prep)\b", r"\b40[- ]?minute\b", r"\b40\s*min\b", r"\bbusy kitchen\b", r"\boverloaded\b", r"\bbacklog\b", r"\bstuck at restaurant\b", r"\bstuck in restaurant\b"],
    "traffic_obstruction": [r"\bstuck in traffic\b", r"\btraffic jam\b", r"\btraffic\b", r"\broad block\b", r"\bcongestion\b", r"\baccident\b", r"\barth\b"],
    "missing_driver": [r"\bdriver missing\b", r"\bdriver offline\b", r"\bno-?show\b", r"\bnot responding\b"],
    "payment_issue": [r"\bpayment failed\b", r"\brefund\b", r"\bbilling\b", r"\bcharge failed\b"],
    "safety_incident": [r"\binjured\b", r"\bassault\b", r"\brobbed\b", r"\bdanger\b", r"\bthreat\b", r"\bdriver assaulted\b", r"\bassaulted\b"],
    "customer_complaint": [r"\bworst\b", r"\bbad service\b", r"\bnot coming on time\b", r"\blate\b"],
    "vehicle_issue": [r"\bflat tire\b", r"\bflat\b", r"\btire\b", r"\bbroken down\b", r"\bbreakdown\b"]
}

# Preprocess / normalization helper (call this at start of detect_problems)
def _normalize_issue_text(text: str) -> str:
    if not text:
        return text
    t = text
    # normalize "40 min" or "40 mins" -> "40-minute"
    t = re.sub(r"\b(\d+)\s*mins?\b", r"\1-minute", t, flags=re.IGNORECASE)
    t = re.sub(r"\b(\d+)\s*min\b", r"\1-minute", t, flags=re.IGNORECASE)
    # normalize common contractions or punctuation that break phrase matching
    t = t.replace("/", " ").replace("&", " and ")
    return t

# helper: test patterns with word boundaries and order (phrases first)
def _keyword_detect_strict(text: str) -> List[Tuple[str, float, List[str]]]:
    """
    Returns list of (label, confidence, matched_patterns_list)
    Confidence is a heuristic: 0.6 base + 0.15 per matched pattern (capped).
    """
    text_l = (text or "").lower()
    hits = []
    for label, patterns in PROBLEM_KEYWORDS_STRICT.items():
        matched = []
        for pat in patterns:
            if re.search(pat, text_l):
                matched.append(pat)
        if matched:
            conf = min(0.95, 0.6 + 0.15 * len(matched))
            hits.append((label, conf, matched))
    return hits

    # normalize input text for better matching
    text = _normalize_issue_text(text)

def detect_problems(text: str, image_metadata: Dict[str,Any]=None, llm_invoke_fn=None) -> List[Dict[str,Any]]:
    """
    New detect_problems using strict keyword matcher + optional LLM augmentation + context adjustments.
    Returns sorted list of dicts: {'label', 'priority', 'confidence', 'reason', 'matched_patterns'}
    """
    kw_hits = _keyword_detect_strict(text)
    llm_hits = []
    if llm_invoke_fn:
        try:
            # keep existing LLM detector if you want (returns list of (label, confidence))
            raw_llm = llm_detect_problems(llm_invoke_fn, text)
            for lab, conf in raw_llm:
                llm_hits.append((lab, float(conf), ["llm"]))
        except Exception:
            llm_hits = []

    # merge hits, prefer higher confidence
    merged = {}
    details = {}
    for label, conf, matched in kw_hits:
        merged[label] = max(merged.get(label, 0.0), conf)
        details[label] = {"matched": matched, "source": "keyword"}
    for label, conf, matched in llm_hits:
        merged[label] = max(merged.get(label, 0.0), conf)
        if label in details:
            details[label]["source"] = "keyword+llm"
            details[label]["matched_llm"] = True
        else:
            details[label] = {"matched": matched, "source": "llm"}

    # context-aware adjustments:
    text_l = (text or "").lower()
    # If 'restaurant' appears near stuck/prep -> boost overloaded_restaurant and reduce traffic
    if re.search(r"\brestaurant\b", text_l) and "traffic_obstruction" in merged:
        # demote traffic if there is restaurant and prep words
        if "overloaded_restaurant" in merged:
            # increase restaurant confidence
            merged["overloaded_restaurant"] = max(merged.get("overloaded_restaurant", 0.6), merged["overloaded_restaurant"] + 0.1)
        # decrease traffic confidence
        merged["traffic_obstruction"] = merged.get("traffic_obstruction", 0.0) - 0.25
        if merged["traffic_obstruction"] <= 0:
            del merged["traffic_obstruction"]
            details.pop("traffic_obstruction", None)

    # boost damaged if image vision says so
    if image_metadata:
        vr = image_metadata.get("vision_result") if isinstance(image_metadata, dict) else None
        if vr and isinstance(vr, dict):
            for k in vr.keys():
                if "damage" in k.lower() or "broken" in k.lower():
                    merged["damaged_item"] = max(merged.get("damaged_item", 0.0), 0.9)
                    details.setdefault("damaged_item", {})["vision_boosted"] = True

    # build result list
    results = []
    for label, conf in merged.items():
        if conf <= 0:
            continue
        pr = PRIORITY_MAP.get(label, PRIORITY_MAP["other"])
        res = {"label": label, "priority": pr, "confidence": round(float(conf), 2), "reason": details.get(label, {}).get("source", "keyword/llm"), "matched_patterns": details.get(label, {}).get("matched", [])}
        results.append(res)

    # sort by priority then confidence
    results.sort(key=lambda x: (x["priority"], -x["confidence"]))
    return results

# Optional LLM detector: if you have an LLM wrapper, you can pass it into detect_problems as llm_invoke_fn
def llm_detect_problems(llm_invoke_fn, text: str) -> List[Tuple[str, float]]:
    prompt = (
        "Extract delivery incident labels from the text. Return JSON array: "
        "[{\"label\":\"...\",\"confidence\":0.0}, ...].\n\nText:\n'''%s'''\n" % text
    )
    raw = llm_invoke_fn(prompt)
    try:
        parsed = json.loads(raw)
        return [(p["label"], float(p.get("confidence", 0.5))) for p in parsed]
    except Exception:
        return []

# ---------------------
# Compound gate helpers (add this right after detect_problems)
# ---------------------
import json
import re
from typing import Tuple, Dict, Any

COMPOUND_KEYWORDS = [
    " and ", " & ", " also ", " plus ", ";", ",", " as well ", "along with",
    "while", "but", "also has", "and the", "&&", "moreover"
]

def _keyword_compound_check(text: str, min_keywords: int = 1) -> Tuple[bool, Dict]:
    text_l = (text or "").lower()
    cnt = sum(1 for kw in COMPOUND_KEYWORDS if kw in text_l)
    # use existing detect_problems to find distinct labels
    try:
        probs = detect_problems(text) if "detect_problems" in globals() else []
        distinct_labels = len(probs)
    except Exception:
        distinct_labels = 0
    meta = {"keyword_count": cnt, "distinct_labels": distinct_labels}
    is_comp = (distinct_labels >= 2) or (cnt >= min_keywords)
    return is_comp, meta

# Optional LLM fallback (only if you have llm_invoke_fn)
LLM_COMPOUND_PROMPT = """
You are a classifier. Input: short user text describing delivery issues.
Return JSON exactly like: {"is_compound": true/false, "confidence": 0.0-1.0, "labels": ["traffic","damaged_item"]}

Now classify the text below. Text: '''{text}'''
"""

def llm_compound_check(llm_invoke_fn, text: str) -> Tuple[bool, Dict]:
    if llm_invoke_fn is None:
        return False, {"reason":"no_llm_fn"}
    prompt = LLM_COMPOUND_PROMPT.format(text=text)
    raw = llm_invoke_fn(prompt)
    try:
        parsed = json.loads(raw)
        return bool(parsed.get("is_compound")), parsed
    except Exception:
        s = (raw or "").lower()
        is_comp = ("true" in s and "false" not in s)
        return is_comp, {"raw": raw}

def is_compound(text: str, llm_invoke_fn=None, debug: bool=False) -> Tuple[bool, Dict[str,Any]]:
    """
    Returns (is_compound, details).
    Strategy:
      1) quick keyword + detect_problems check
      2) if ambiguous -> optional llm_invoke_fn fallback
    """
    details = {}
    h_comp, h_meta = _keyword_compound_check(text)
    details["heuristic"] = h_meta
    if debug:
        print("Heuristic:", h_comp, h_meta)

    # strong heuristic: >1 distinct label
    if h_meta.get("distinct_labels", 0) >= 2:
        details["final_source"] = "heuristic_labels"
        return True, details
    if h_meta.get("keyword_count", 0) >= 2:
        details["final_source"] = "heuristic_keywords"
        return True, details

    # ambiguous -> try LLM (if provided)
    if llm_invoke_fn is not None:
        llm_flag, llm_meta = llm_compound_check(llm_invoke_fn, text)
        details["llm"] = llm_meta
        conf = float(llm_meta.get("confidence", 0))
        if conf >= 0.65:
            details["final_source"] = "llm"
            return bool(llm_meta.get("is_compound")), details

    details["final_source"] = "heuristic_default"
    return False, details


# Utility to determine if function is coroutine
def _is_coroutine(fn):
    return asyncio.iscoroutinefunction(fn)

async def _call_handler_async(fn, *args, **kwargs):
    if _is_coroutine(fn):
        return await fn(*args, **kwargs)
    else:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, lambda: fn(*args, **kwargs))

async def resolve_compound(problems: List[Dict[str,Any]], handlers_map: Dict[str, Any],
                     policy: str = "priority", parallel_allow: List[str]=None,
                     case_context: Dict[str,Any]=None) -> Dict[str, Any]:
    if parallel_allow is None:
        parallel_allow = ["traffic_obstruction", "overloaded_restaurant", "customer_complaint"]

    log = []
    results = {}
    problems_sorted = sorted(problems, key=lambda x: (x["priority"], -x["confidence"]))
    log.append({"step": "sorted_problems", "value": problems_sorted})

    if policy == "parallel":
        parallel_group = [p for p in problems_sorted if p["label"] in parallel_allow]
        sequential_group = [p for p in problems_sorted if p["label"] not in parallel_allow]
        if parallel_group:
            tasks = []
            for p in parallel_group:
                handler = handlers_map.get(p["label"])
                if handler:
                    tasks.append(_call_handler_async(handler, p, case_context))
                else:
                    log.append({"step":"no_handler", "label":p["label"]})
            if tasks:
                loop = asyncio.get_event_loop()
                done = await asyncio.gather(*tasks)
                for p, r in zip(parallel_group, done):
                    results[p["label"]] = r
                    log.append({"step": "parallel_done", "label": p["label"], "result": r})
        for p in sequential_group:
            handler = handlers_map.get(p["label"])
            if handler:
                r = asyncio.get_event_loop().run_until_complete(_call_handler_async(handler, p, case_context))
                results[p["label"]] = r
                log.append({"step":"sequential_done","label":p["label"], "result": r})
            else:
                log.append({"step":"no_handler", "label":p["label"]})
    else:
        for p in problems_sorted:
            handler = handlers_map.get(p["label"])
            if handler:
                log.append({"step":"invoking", "label": p["label"], "priority": p["priority"]})
                r = await _call_handler_async(handler, p, case_context)
                results[p["label"]] = r
                log.append({"step":"done", "label":p["label"], "result": r})
                if isinstance(r, dict) and r.get("stop_chain", False):
                    log.append({"step":"stopped_by_handler", "label": p["label"]})
                    break
            else:
                log.append({"step":"no_handler", "label":p["label"]})
    return {"results": results, "log": log, "processed": [p["label"] for p in problems_sorted]}

# ===== Default handlers adapted to your notebook function names =====
# Replace wrapper internals if your functions have different args / return types.

def _wrap_notify(msg: str):
    try:
        # your notebook has notify_customer / notify
        if 'notify_customer' in globals():
            return notify_customer(msg)
        elif 'notify' in globals():
            return notify(msg)
        else:
            print("Notification:", msg)
            return {"status":"notified", "msg": msg}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_traffic(p, ctx):
    # call check_traffic or calculate_alternative_route if available
    try:
        # first attempt your calculate_alternative_route() if present
        if 'calculate_alternative_route' in globals():
            alt = calculate_alternative_route(ctx or {})
            _wrap_notify(f"Traffic detected. Suggested reroute; ETA updated.")
            return {"status":"routed", "detail": alt}
        elif 'check_traffic' in globals():
            t = check_traffic(ctx or {})
            _wrap_notify("Traffic noted; taking action.")
            return {"status":"checked", "detail": t}
        else:
            _wrap_notify("Traffic detected; no route function available.")
            return {"status":"no_op"}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_overloaded_restaurant(p, ctx):
    try:
        _wrap_notify("Restaurant overloaded. Suggest alternatives to customer and notify merchant.")
        return {"status":"merchant_notified"}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_damaged_item(p, ctx):
    try:
        # analyze_evidence is present in your notebook
        vision_res = None
        if 'analyze_evidence' in globals():
            vision_res = analyze_evidence(ctx.get("evidence")) if ctx else analyze_evidence(None)
        # if refund exists, call it
        if vision_res and isinstance(vision_res, dict) and vision_res.get("damage_confidence", 0) > 0.75:
            if 'refund' in globals():
                refund_res = refund(p, ctx)
                _wrap_notify("Damage confirmed; refund initiated.")
                return {"status":"refund_initiated", "detail": refund_res, "stop_chain": True}
        # fallback: escalate if uncertain
        if 'escalate_to_human_agent' in globals():
            escalate_to_human_agent(p, ctx)
            _wrap_notify("Damage suspected; escalated to human.")
            return {"status":"escalated", "stop_chain": True}
        _wrap_notify("Damage reported; queued for manual review.")
        return {"status":"queued_manual_review"}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_safety_incident(p, ctx):
    try:
        if 'escalate_to_human_agent' in globals():
            escalate_to_human_agent(p, ctx)
        _wrap_notify("Safety incident detected. Escalating to human team with high priority.")
        return {"status":"escalated_high", "stop_chain": True}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_missing_driver(p, ctx):
    try:
        _wrap_notify("Driver missing — dispatching backup.")
        return {"status":"dispatch_backup"}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_payment_issue(p, ctx):
    try:
        _wrap_notify("Payment issue: initiating finance workflow.")
        return {"status":"finance_started"}
    except Exception as e:
        return {"status":"error","error": str(e)}

def _handler_customer_complaint(p, ctx):
    try:
        _wrap_notify("Customer complaint logged; sending apology + ETA.")
        return {"status":"logged"}
    except Exception as e:
        return {"status":"error","error": str(e)}

DEFAULT_HANDLERS = {
    "traffic_obstruction": _handler_traffic,
    "overloaded_restaurant": _handler_overloaded_restaurant,
    "damaged_item": _handler_damaged_item,
    "safety_incident": _handler_safety_incident,
    "missing_driver": _handler_missing_driver,
    "payment_issue": _handler_payment_issue,
    "customer_complaint": _handler_customer_complaint,
    # map vehicle to traffic handler by default
    "vehicle_issue": _handler_traffic,
}


# ===== patched handle_text_case (replace your existing function with this) =====
async def handle_text_case(text: str, image_metadata: Dict[str, Any] = None, llm_invoke_fn=None, handlers_map: Dict[str, Any] = None, policy='priority'):
    """
    Reworked dispatcher:
     - Uses is_compound() to gate single vs compound.
     - For single: dispatch to runner_map (sync or async) via _call_handler_async.
     - If no top_label found: try llm_detect_problems as fallback to get a top label.
     - If still no label or no runner mapped: fall back to resolve_compound for safe handling.
     - Shows clear agent panels and avoids duplicate printing.
    """
    if handlers_map is None:
        handlers_map = DEFAULT_HANDLERS

    # Runner map: label -> runner function (fill with your run_* functions)
    runner_map = {
        "damaged_item": globals().get("run_damage_only"),
        "traffic_obstruction": globals().get("run_traffic_only"),
        "overloaded_restaurant": globals().get("run_overloaded_restaurant_only"),
        "safety_incident": globals().get("run_safety_only"),   # optional: add run_safety_only if present
    }

    # 1) Gate decision
    gate_flag, gate_meta = is_compound(text, llm_invoke_fn=llm_invoke_fn, debug=False)
    show_agent_response({"gate_decision": gate_flag, "meta": gate_meta})

    # 2) SINGLE flow
    if not gate_flag:
        probs = detect_problems(text, image_metadata=image_metadata, llm_invoke_fn=llm_invoke_fn)
        top_label = probs[0]["label"] if probs else None
        show_agent_response({"dispatch": "single", "top_label": top_label, "detected": probs})

        # If no top_label from heuristic, try LLM-provided labels (llm_detect_problems)
        if top_label is None and llm_invoke_fn is not None:
            try:
                llm_raw = llm_detect_problems(llm_invoke_fn, text)  # returns list of (label, confidence)
                if llm_raw:
                    top_label = llm_raw[0][0]
                    show_agent_response({"note": "LLM fallback provided top_label", "llm_top": top_label, "llm_conf": llm_raw[0][1]})
            except Exception as e:
                show_warning(f"LLM fallback failed: {e}")

        # If we now have a top_label and a mapped runner, call it
        if top_label:
            runner = runner_map.get(top_label)
            if runner:
                try:
                    # Call runner via _call_handler_async (works for sync or async functions)
                    if top_label == "damaged_item":
                        # run_damage_only(user_input, order_id=None, image_path=None)
                        payload = await _call_handler_async(runner, text, None, image_metadata)
                    elif top_label == "traffic_obstruction":
                        # run_traffic_only(user_input, location=None, start=None, end=None)
                        payload = await _call_handler_async(runner, text)
                    elif top_label == "overloaded_restaurant":
                        # try to extract context for merchant/driver if present
                        ctx = probs[0].get("context", {}) if probs else {}
                        payload = await _call_handler_async(runner,
                                                           ctx.get("merchant_id", "M123"),
                                                           ctx.get("driver_id", "DRV101"),
                                                           ctx.get("cuisine", "Food"),
                                                           ctx.get("prep_time", None))
                    elif top_label == "customer_complaint" and 'handle_customer_delay_complaint' in globals():
                        payload = await _call_handler_async(lambda: handle_customer_delay_complaint(text))
                    else:
                        # generic call
                        payload = await _call_handler_async(runner, text)
                except Exception as e:
                    show_warning(f"Runner execution error for {top_label}: {e}")
                    payload = {"error": str(e)}
            else:
                # no mapped runner for the label
                show_agent_response({"note": f"No runner mapped for label '{top_label}'. Falling back to compound resolver."})
                if any(p["label"]=="damaged_item" for p in probs) and any(p["label"]=="traffic_obstruction" for p in probs):
                  resolution = await resolve_compound(probs, handlers_map, policy="parallel", parallel_allow=["traffic_obstruction"])
                else:
                  resolution = await resolve_compound(probs, handlers_map, policy=policy, case_context={"evidence": image_metadata})
                payload = {"combined_summary": "fallback_compound", "resolution": resolution}
        else:
            # still no label after LLM fallback -> safe fallback to compound resolver
            show_agent_response("No label detected after heuristics and LLM fallback. Using compound resolver fallback.")
            if any(p["label"]=="damaged_item" for p in probs) and any(p["label"]=="traffic_obstruction" for p in probs):
                  resolution = await resolve_compound(probs, handlers_map, policy="parallel", parallel_allow=["traffic_obstruction"])
            else:
                  resolution = await resolve_compound(probs, handlers_map, policy=policy, case_context={"evidence": image_metadata})
            payload = {"combined_summary": "fallback_compound", "resolution": resolution}

        # Display a concise result summary (only once)
        if isinstance(payload, dict) and payload.get("combined_summary"):
            show_agent_response(payload.get("combined_summary"))
        else:
            show_agent_response(str(payload))
        return payload

    # 3) COMPOUND flow
    else:
        probs = detect_problems(text, image_metadata=image_metadata, llm_invoke_fn=llm_invoke_fn)
        show_agent_response({"dispatch": "compound", "detected": probs})

        resolution = await resolve_compound(probs, handlers_map, policy=policy, case_context={"evidence": image_metadata})

        # Render resolver log: tool calls and results
        for entry in resolution.get("log", []):
            step = entry.get("step")
            if step == "invoking":
                show_tool_call(entry.get("label"), {"priority": entry.get("priority"), "note": entry.get("note", None)})
            elif step in ("done", "parallel_done", "sequential_done"):
                show_tool_result(entry.get("label"), entry.get("result"))
            else:
                show_thinking(step, entry)

        # Final combined summary (human-friendly)
        combined_summary = "Compound resolution finished."
        try:
            if isinstance(resolution.get("results"), dict) and resolution.get("results"):
                combined_summary = ", ".join([f"{k}:{v.get('status','')}" for k, v in resolution.get("results").items()])
        except Exception:
            pass

        show_agent_response(combined_summary)
        return {"combined_summary": combined_summary, "resolution": resolution}
# ===== end patched handle_text_case =====

In [ ]:
await handle_text_case("Traffic jam at airport road. Driver stuck in jam.")
await handle_text_case("Customer reports food is damaged; driver stuck at restaurant with 40-minute prep.")
await handle_text_case("Driver assaulted and food stolen during delivery.")

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 0, 'distinct_labels': 1}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{
    'dispatch': 'single',
    'top_label': 'traffic_obstruction',
    'detected': [
        {
            'label': 'traffic_obstruction',
            'priority': 4,
            'confidence': 0.9,
            'reason': 'keyword',
            'matched_patterns': ['\\btraffic jam\\b', '\\btraffic\\b']
        }
    ]
}

GRABNAVI RESPONSE

...

{'note': "No runner mapped for label 'safety_incident'. Falling back to compound resolver."}

Escalating to human agent. Problem: {'label': 'safety_incident', 'priority': 1, 'confidence': 0.9, 'reason': 'keyword', 'matched_patterns': ['\\bdriver assaulted\\b', '\\bassaulted\\b']}, Context: {'evidence': None}
Notify Customer: Safety incident detected. Escalating to human team with high priority.


GRABNAVI RESPONSE

fallback_compound

{'combined_summary': 'fallback_compound',
 'resolution': {'results': {'safety_incident': {'status': 'escalated_high',
    'stop_chain': True}},
  'log': [{'step': 'sorted_problems',
    'value': [{'label': 'safety_incident',
      'priority': 1,
      'confidence': 0.9,
      'reason': 'keyword',
      'matched_patterns': ['\\bdriver assaulted\\b', '\\bassaulted\\b']}]},
   {'step': 'invoking', 'label': 'safety_incident', 'priority': 1},
   {'step': 'done',
    'label': 'safety_incident',
    'result': {'status': 'escalated_high', 'stop_chain': True}},
   {'step': 'stopped_by_handler', 'label': 'safety_incident'}],
  'processed': ['safety_incident']}}

In [ ]:
import gradio as gr

# Wrapper for Gradio to interact with async handle_text_case
def driver_chat(user_message, history):
    try:
        # Run your async handle_text_case inside event loop
        loop = asyncio.get_event_loop()
        result = loop.run_until_complete(handle_text_case(user_message))

        # Convert result to readable text for driver
        if isinstance(result, dict):
            reply = str(result.get("combined_summary", result))
        else:
            reply = str(result)

        # Return updated chat history
        history.append((user_message, reply))
        return history, ""
    except Exception as e:
        history.append((user_message, f"[ERROR] {str(e)}"))
        return history, ""


In [ ]:
# === LEGACY_COMPOUND_HANDLING_BACKUP ===

# Responsibilities:
# 1. Detect & prioritize multiple incident types (traffic, damage, overload).
# 2. Execute cases sequentially, avoiding duplicate notifications.
# 3. Generate combined summary + structured artifacts:
#    - trace.json : Full execution trace
#    - trace.csv  : Tabular summary (per-incident)
# ============================================================

# ------------------------------------------------------------
# Priority Mapping for Incident Types
# - Lower number = higher priority
# ------------------------------------------------------------

PRIORITY = {
    "Traffic Obstruction": 1,
    "Overloaded Restaurant": 2,
    "Damaged Package": 3,
    "Unknown": 99
}


# ------------------------------------------------------------
# Summarize a Single Trace (for readability in final output)
# ------------------------------------------------------------

def _summarize_single_trace(t: Dict[str, Any]) -> str:
    tools = [s["tool"] for s in t["steps"]]
    if t["label"] == "Traffic Obstruction":
        parts = []
        if "check_traffic" in tools: parts.append("traffic checked")
        if "calculate_alternative_route" in tools: parts.append("reroute calculated")
        if "notify_user" in tools: parts.append("customer notified")
        return "Traffic: " + ", ".join(parts) + "."
    if t["label"] == "Damaged Package":
        parts = []
        if "collect_evidence" in tools: parts.append("evidence collected")
        if "analyze_evidence" in tools:
            try:
                obs = t["steps"][1]["observation"]
                dmg = (obs.get("damage_detected") if isinstance(obs, dict) else json.loads(obs).get("damage_detected"))
                conf = (obs.get("confidence") if isinstance(obs, dict) else json.loads(obs).get("confidence"))
                parts.append(f"analysis={bool(dmg)} (conf={float(conf):.2f})")
            except Exception:
                parts.append("analysis done")
        if "issue_instant_refund" in tools: parts.append("refund issued")
        if "exonerate_driver" in tools: parts.append("driver cleared")
        if "notify_user" in tools: parts.append("customer notified")
        return "Damage: " + ", ".join(parts) + "."
    if t["label"] == "Overloaded Restaurant":
        parts = []
        if "get_merchant_status" in tools: parts.append("status checked")
        if "notify_customer" in tools: parts.append("customer notified + voucher")
        if "re_route_driver" in tools: parts.append("driver rerouted")
        if "get_nearby_merchants" in tools: parts.append("alternatives suggested")
        return "Overloaded Restaurant: " + ", ".join(parts) + "."
    return f"{t['label']}: handled."
# ------------------------------------------------------------
# Run Compound Case with Full Trace
# ------------------------------------------------------------
def run_compound_case_with_trace(user_input: str):
      # 1. Detect incidents from input
    incidents = detect_incidents_llm(user_input)
    if not incidents:
        incidents = [{"label": "Unknown", "fields": {}}]
        # 2. Sort incidents by priority
    incidents = sorted(incidents, key=lambda i: PRIORITY.get(i.get("label","Unknown"), 99))

 # 3. Prepare agent + storage
    agent = make_agent(return_steps=True)
    all_traces: List[Dict[str, Any]] = []
    seen_notifications: Set[Tuple[str, str]] = set()

# 4. Run incidents sequentially
    for inc in incidents:
        label  = inc["label"]
        fields = inc.get("fields", {})

          # Route to appropriate handler
        if label == "Traffic Obstruction":
            trace = handle_traffic_case(agent, user_input, fields)
        elif label == "Damaged Package":
            trace = handle_damage_case(agent, user_input, fields)
        elif label == "Overloaded Restaurant":
            trace = handle_overloaded_restaurant_case(agent, fields)
        else:
            res = agent.invoke({"input": "Notify user: Unable to classify; escalating to human support."})
            actions = extract_actions(res.get("intermediate_steps", []), "Unknown")
            trace = {"label": "Unknown", "context": {}, "steps": actions,
                     "resolution": f"Unknown issue handled in {len(actions)} steps"}

        # Deduplicate identical notify messages to customer
        deduped = []
        for a in trace["steps"]:
            if a["tool"] in {"notify_user","notify_customer"}:
                payload = a["observation"]
                if isinstance(payload, str):
                    try: payload = json.loads(payload)
                    except: payload = {}
                role = payload.get("role", "customer")
                msg  = payload.get("message")
                key = (role, msg)
                if key and key not in seen_notifications:
                    seen_notifications.add(key)
                    deduped.append(a)
            else:
                deduped.append(a)
        trace["steps"] = deduped
        all_traces.append(trace)

    # 5. Build combined summary across all incidents
    combined_summary = " | ".join(_summarize_single_trace(t) for t in all_traces)


    # 6. Generate artifacts (JSON + CSV)
    summary_rows = [{
        "label": t["label"],
        "resolution": t["resolution"],
        "steps": len(t["steps"]),
        "context": json.dumps(t.get("context", {}))
    } for t in all_traces]

    with open("trace.json", "w") as f:
        json.dump({"combined_summary": combined_summary, "traces": all_traces}, f, indent=2)
    pd.DataFrame(summary_rows).to_csv("trace.csv", index=False)

    # 7. Print + display results
    print(json.dumps({"combined_summary": combined_summary, "traces": all_traces}, indent=2))
    try:
        display(pd.DataFrame(summary_rows))
    except Exception:
        pass

    return {"combined_summary": combined_summary, "traces": all_traces}

# ------------------------------------------------------------
#  Backward Compatibility Alias
# ------------------------------------------------------------
def run_case_with_trace(user_input: str, label: str = ""):
    return run_compound_case_with_trace(user_input)


In [ ]:
# === COMPOUND_PROBLEM_HANDLING_EXAMPLES ===
print(await handle_text_case("Traffic jam at airport road. Driver stuck in jam."))
print(await handle_text_case("Customer reports food is damaged; driver stuck at restaurant with 40-minute prep."))
print(await handle_text_case("Driver assaulted and food stolen during delivery."))

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 0, 'distinct_labels': 1}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{
    'dispatch': 'single',
    'top_label': 'traffic_obstruction',
    'detected': [
        {
            'label': 'traffic_obstruction',
            'priority': 4,
            'confidence': 0.9,
            'reason': 'keyword',
            'matched_patterns': ['\\btraffic jam\\b', '\\btraffic\\b']
        }
    ]
}

GRABNAVI RESPONSE

...

{'note': "No runner mapped for label 'safety_incident'. Falling back to compound resolver."}

Escalating to human agent. Problem: {'label': 'safety_incident', 'priority': 1, 'confidence': 0.9, 'reason': 'keyword', 'matched_patterns': ['\\bdriver assaulted\\b', '\\bassaulted\\b']}, Context: {'evidence': None}
Notify Customer: Safety incident detected. Escalating to human team with high priority.


GRABNAVI RESPONSE

fallback_compound

{'combined_summary': 'fallback_compound', 'resolution': {'results': {'safety_incident': {'status': 'escalated_high', 'stop_chain': True}}, 'log': [{'step': 'sorted_problems', 'value': [{'label': 'safety_incident', 'priority': 1, 'confidence': 0.9, 'reason': 'keyword', 'matched_patterns': ['\\bdriver assaulted\\b', '\\bassaulted\\b']}]}, {'step': 'invoking', 'label': 'safety_incident', 'priority': 1}, {'step': 'done', 'label': 'safety_incident', 'result': {'status': 'escalated_high', 'stop_chain': True}}, {'step': 'stopped_by_handler', 'label': 'safety_incident'}], 'processed': ['safety_incident']}}


# Upload Evidence Image

In [ ]:

# Purpose:
# - Allows the user to upload a local image (via Colab file picker).
# - Saves uploaded file path into global `EVIDENCE_IMAGE`.
# - This image will later be used in:
#     → Evidence Collection
#     → Image Analysis (e.g., damaged package detection).
# ============================================================

# ------------------------------------------------------------
#  Upload an image & update global evidence path
# ------------------------------------------------------------
# ============================================================
# Purpose:
# - Allows the system to use an image directly from a URL.
# - No runtime upload needed.
# - Saves downloaded file into global EVIDENCE_IMAGE.
# - This image will later be used in:
#     → Evidence Collection
#     → Image Analysis (e.g., damaged package detection).
# ============================================================

import requests

# ------------------------------------------------------------
#  Insert evidence image from a fixed URL (compile-time)
# ------------------------------------------------------------

# Example: Replace this with your actual image URL
IMAGE_URL = "https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.istockphoto.com%2Fphotos%2Fdamaged-parcel&psig=AOvVaw0alsggb86azQVYgaaUF_d9&ust=1757180500516000&source=images&cd=vfe&opi=89978449&ved=0CBUQjRxqFwoTCJjF4NWVwo8DFQAAAAAdAAAAABAy"
EVIDENCE_IMAGE = "https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.istockphoto.com%2Fphotos%2Fdamaged-parcel&psig=AOvVaw0alsggb86azQVYgaaUF_d9&ust=1757180500516000&source=images&cd=vfe&opi=89978449&ved=0CBUQjRxqFwoTCJjF4NWVwo8DFQAAAAAdAAAAABAy"
print("EVIDENCE_IMAGE set to:", EVIDENCE_IMAGE)

EVIDENCE_IMAGE set to: https://www.google.com/url?sa=i&url=https%3A%2F%2Fwww.istockphoto.com%2Fphotos%2Fdamaged-parcel&psig=AOvVaw0alsggb86azQVYgaaUF_d9&ust=1757180500516000&source=images&cd=vfe&opi=89978449&ved=0CBUQjRxqFwoTCJjF4NWVwo8DFQAAAAAdAAAAABAy


# Run Trace Compound

In [ ]:
# ------------------------------------------------------------
# 1️⃣ Traffic Obstruction Demo
# ------------------------------------------------------------
# Simulates a driver reporting a traffic jam on MainRoad
# while heading to the Airport from Restaurant.
# (Uncomment to run)
# run_compound_case_with_trace(
#     "Driver reports jam on MainRoad while going to Airport from Restaurant."
# )

# ------------------------------------------------------------
# 2️⃣ Damaged Package Demo
# ------------------------------------------------------------
# Simulates a customer reporting a damaged order.
# Relies on your uploaded EVIDENCE_IMAGE.
# (Uncomment to run)
# run_compound_case_with_trace(
#     "Customer says package was damaged. Order ID: ORD777."
# )

# ------------------------------------------------------------
# 3️⃣ Compound Case Demo (Traffic + Damage)
# ------------------------------------------------------------
# Simulates a situation with both:
#   - Traffic jam on MainRoad
#   - Package damage (bag open, contents exposed)
# Demonstrates orchestrator handling multiple incident types together.
run_compound_case_with_trace(
    "Stuck in traffic on MainRoad to Airport from Restaurant and the order bag is open and contents exposed. Order ID: ORD123."
)




> Entering new AgentExecutor chain...

Invoking: `check_traffic` with `{'location': 'MainRoad'}`


{"location": "MainRoad", "incident": "accident", "delay_minutes": 18}
Invoking: `calculate_alternative_route` with `{'end': 'Airport', 'start': 'Restaurant'}`


{"start": "Restaurant", "end": "Airport", "path": ["Restaurant", "MainRoad", "Airport"], "eta_minutes": 26}
Invoking: `notify_user` with `{'message': 'Traffic incident detected at MainRoad. Rerouting...', 'role': 'customer'}`


{"role": "customer", "message": "Traffic incident detected at MainRoad. Rerouting..."}OK. I have rerouted your order. Expect a delay of 26 minutes.

> Finished chain.


/tmp/ipython-input-1018765942.py:152: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z"
/tmp/ipython-input-1042956962.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"order_id": oid}, "observation": ev, "ts": datetime.utcnow().isoformat() + "Z"


{
  "combined_summary": "Traffic: traffic checked, reroute calculated, customer notified. | Damage: evidence collected, analysis=False (conf=0.55), customer notified.",
  "traces": [
    {
      "label": "Traffic Obstruction",
      "context": {
        "location": "MainRoad",
        "start": "Restaurant",
        "end": "Airport"
      },
      "steps": [
        {
          "step": 1,
          "incident": "Traffic Obstruction",
          "tool": "check_traffic",
          "args": {
            "location": "MainRoad"
          },
          "observation": "{\"location\": \"MainRoad\", \"incident\": \"accident\", \"delay_minutes\": 18}",
          "ts": "2025-09-09T19:31:51.287789Z"
        },
        {
          "step": 2,
          "incident": "Traffic Obstruction",
          "tool": "calculate_alternative_route",
          "args": {
            "end": "Airport",
            "start": "Restaurant"
          },
          "observation": "{\"start\": \"Restaurant\", \"end\": \"Airport\"

/tmp/ipython-input-1042956962.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "observation": ana, "ts": datetime.utcnow().isoformat() + "Z"
/tmp/ipython-input-1042956962.py:89: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"role": "customer", "message": message}, "observation": note, "ts": datetime.utcnow().isoformat() + "Z"


,label,resolution,steps,context
0,Traffic Obstruction,Traffic Obstruction handled in 3 steps,3,"{""location"": ""MainRoad"", ""start"": ""Restaurant""..."
1,Damaged Package,Damaged Package handled in 3 steps,3,"{""order_id"": ""ORDER""}"


{'combined_summary': 'Traffic: traffic checked, reroute calculated, customer notified. | Damage: evidence collected, analysis=False (conf=0.55), customer notified.',
 'traces': [{'label': 'Traffic Obstruction',
   'context': {'location': 'MainRoad',
    'start': 'Restaurant',
    'end': 'Airport'},
   'steps': [{'step': 1,
     'incident': 'Traffic Obstruction',
     'tool': 'check_traffic',
     'args': {'location': 'MainRoad'},
     'observation': '{"location": "MainRoad", "incident": "accident", "delay_minutes": 18}',
     'ts': '2025-09-09T19:31:51.287789Z'},
    {'step': 2,
     'incident': 'Traffic Obstruction',
     'tool': 'calculate_alternative_route',
     'args': {'end': 'Airport', 'start': 'Restaurant'},
     'observation': '{"start": "Restaurant", "end": "Airport", "path": ["Restaurant", "MainRoad", "Airport"], "eta_minutes": 26}',
     'ts': '2025-09-09T19:31:51.287816Z'},
    {'step': 3,
     'incident': 'Traffic Obstruction',
     'tool': 'notify_user',
     'args': {'m

# Single Run Trace

In [ ]:
# Purpose:
# - Provide separate runner functions for each individual case type:
#     1. Traffic Obstruction (ignores damage)
#     2. Damaged Package (uses uploaded or overridden EVIDENCE_IMAGE)
#     3. Overloaded Restaurant (merchant cannot handle more orders)
#
# Common Features:
# - Each runner executes only its respective flow.
# - Results are:
#     * Summarized into a human-readable string
#     * Saved as JSON (trace_single.json)
#     * Saved as CSV (trace_single.csv)
# - Outputs also displayed in a DataFrame (if possible, for Colab readability).
# ============================================================

from typing import Dict, Any, Optional, List, Tuple, Set
import json
import pandas as pd
from IPython.display import display
from datetime import datetime


# ------------------------------------------------------------
#  Helper: Summarize a single-case trace
# ------------------------------------------------------------
def _single_summary(trace: Dict[str, Any]) -> str:
    """
    Convert an execution trace into a concise summary string.
    Includes key actions performed depending on the case label.
    """
    tools = [s["tool"] for s in trace["steps"]]

    # --- Traffic Obstruction ---
    if trace["label"] == "Traffic Obstruction":
        parts = []
        if "check_traffic" in tools: parts.append("traffic checked")
        if "calculate_alternative_route" in tools: parts.append("reroute calculated")
        if "notify_user" in tools: parts.append("customer notified")
        return "Traffic: " + ", ".join(parts) + "."


    # --- Damaged Package ---
    if trace["label"] == "Damaged Package":
        parts = []
        if "collect_evidence" in tools: parts.append("evidence collected")
        if "analyze_evidence" in tools:
            try:
                obs = trace["steps"][1]["observation"]
                if isinstance(obs, str):
                    obs = json.loads(obs)
                parts.append(f"analysis={bool(obs.get('damage_detected'))} (conf={float(obs.get('confidence',0)):0.2f})")
            except Exception:
                parts.append("analysis done")
        if "issue_instant_refund" in tools: parts.append("refund issued")
        if "exonerate_driver" in tools: parts.append("driver cleared")
        if "notify_user" in tools: parts.append("customer notified")
        return "Damage: " + ", ".join(parts) + "."

    # --- Overloaded Restaurant ---
    if trace["label"] == "Overloaded Restaurant":
        parts = []
        if "get_merchant_status" in tools: parts.append("status checked")
        if "notify_customer" in tools: parts.append("customer notified + voucher")
        if "re_route_driver" in tools: parts.append("driver rerouted")
        if "get_nearby_merchants" in tools: parts.append("alternatives suggested")
        return "Overloaded Restaurant: " + ", ".join(parts) + "."

    # --- Default fallback ---
    return f"{trace['label']}: handled."


# ------------------------------------------------------------
# 1. Traffic-Only Runner
# ------------------------------------------------------------
def run_traffic_only(user_input: str, location: Optional[str] = None,
                     start: Optional[str] = None, end: Optional[str] = None) -> Dict[str, Any]:
    agent = make_agent(return_steps=True)
    fields = {"location": location, "start": start, "end": end}
    trace = handle_traffic_case(agent, user_input, fields)

    # 🧠 Show reasoning
    show_thinking("TRAFFIC CASE THINKING", {"input": user_input, "fields": fields})

    # 🔎 Show each step
    for step in trace.get("steps", []):
        show_tool_call(step.get("tool"), step.get("input"))
        show_tool_result(step.get("tool"), step.get("observation"))

    # ✅ Summary
    combined_summary = _single_summary(trace)
    show_agent_response(combined_summary)

    return {"combined_summary": combined_summary, "traces": [trace]}

# ------------------------------------------------------------
# 2️. Damaged Package Runner
# ------------------------------------------------------------
def run_damage_only(user_input: str, order_id: Optional[str] = None, image_path: Optional[str] = None) -> Dict[str, Any]:
    agent = make_agent(return_steps=True)
    fields = {"order_id": order_id}

    global EVIDENCE_IMAGE
    _old_img = EVIDENCE_IMAGE
    if image_path:
        EVIDENCE_IMAGE = image_path

    try:
        trace = handle_damage_case(agent, user_input, fields)
    finally:
        EVIDENCE_IMAGE = _old_img

    # 🧠 Show reasoning
    show_thinking("DAMAGE CASE THINKING", {"input": user_input, "order_id": order_id})

    # 🔎 Steps
    for step in trace.get("steps", []):
        show_tool_call(step.get("tool"), step.get("input"))
        show_tool_result(step.get("tool"), step.get("observation"))

    # ✅ Summary
    combined_summary = _single_summary(trace)
    show_agent_response(combined_summary)

    return {"combined_summary": combined_summary, "traces": [trace]}

# ---------------------------
# Small safety-only runner
# ---------------------------
def run_safety_only(user_input: str, **kwargs):
    """
    Simple flow for safety incidents. Escalates immediately.
    Returns a payload similar to other run_* functions.
    """
    show_thinking("SAFETY CASE THINKING", {"input": user_input})
    # escalate
    if 'escalate_to_human_agent' in globals():
        try:
            escalate_to_human_agent(user_input, kwargs)
        except Exception:
            # best-effort escalate
            pass
    payload = {"combined_summary": "safety incident escalated to human", "traces": []}
    # do NOT print raw JSON here — let handle_text_case display it
    return payload

# ------------------------------------------------------------
# 3️. Overloaded Restaurant Runner
# ------------------------------------------------------------
def run_overloaded_restaurant_only(merchant_id: str, driver_id: str = "DRV101",
                                   cuisine: str = "Food", prep_time_hint: Optional[int] = None) -> Dict[str, Any]:
    agent = make_agent(return_steps=True)
    fields = {
        "merchant_id": merchant_id,
        "driver_id": driver_id,
        "cuisine": cuisine,
        "prep_time": prep_time_hint or 40
    }
    trace = handle_overloaded_restaurant_case(agent, fields)

    # 🧠 Show reasoning
    show_thinking("RESTAURANT CASE THINKING", fields)

    # 🔎 Steps
    for step in trace.get("steps", []):
        show_tool_call(step.get("tool"), step.get("input"))
        show_tool_result(step.get("tool"), step.get("observation"))

    # ✅ Summary
    combined_summary = _single_summary(trace)
    show_agent_response(combined_summary)

    return {"combined_summary": combined_summary, "traces": [trace]}

In [ ]:
# ============================================================
# Run Single-case Examples (Traffic-only / Damage-only)
# - Showcases how to run isolated incident flows
# - Covers:
#   1. Traffic-only runner
#   2. Damage-only runner (with default or custom image)
# ============================================================

# 🚦 TRAFFIC-ONLY DEMO
# ------------------------------------------------------------
# Runs ONLY the traffic obstruction flow.
# Ignores any mentions of damage (package, evidence, etc).
# Example: Reports jam on MainRoad while heading to Airport.
# To test, just uncomment the line below:
# run_traffic_only("Jam on MainRoad from Restaurant to Airport.")


# 📦 DAMAGE-ONLY DEMO (default EVIDENCE_IMAGE)
# ------------------------------------------------------------
# Runs ONLY the damaged package flow.
# Uses the globally configured `EVIDENCE_IMAGE`
# (either uploaded via the Upload cell or default placeholder).
# Example: Order ID = ORD555

await handle_text_case(
    "Package was torn. Order ID: ORD555"
)

# 📸 DAMAGE-ONLY DEMO (with specific uploaded image)
# ------------------------------------------------------------
# Allows testing with a custom photo.
# Pass `image_path="/content/your_file.jpg"`
# → Temporarily overrides global `EVIDENCE_IMAGE` for this run only.
# Example usage (uncomment and replace file name to test):
# run_damage_only(
#     "Package looks open, please check. Order ID: ORD888",
#     image_path="/content/your_photo.jpg"
# )

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 0, 'distinct_labels': 0}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{'dispatch': 'single', 'top_label': None, 'detected': []}

GRABNAVI RESPONSE

No label detected after heuristics and LLM fallback. Using compound resolver fallback.

GRABNAVI RESPONSE

fallback_compound

{'combined_summary': 'fallback_compound',
 'resolution': {'results': {},
  'log': [{'step': 'sorted_problems', 'value': []}],
  'processed': []}}

In [ ]:
# ============================================================
#  Wrong Flow Trigger Example (Damage-only)
# - This runs the Damage-only handler, but the input text
#   describes a *kitchen prep time delay* (restaurant issue),
#   not actual package damage.
# - Useful to see how the pipeline responds when given
#   mismatched input (damage flow forced on delay text).
# ============================================================
await handle_text_case(
    "40-minute kitchen prep time. Order ID: ORD555"
)

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 0, 'distinct_labels': 1}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{
    'dispatch': 'single',
    'top_label': 'overloaded_restaurant',
    'detected': [
        {
            'label': 'overloaded_restaurant',
            'priority': 3,
            'confidence': 0.9,
            'reason': 'keyword',
            'matched_patterns': ['\\b(long prep|prep time|prep)\\b', '\\b40[- ]?minute\\b']
        }
    ]
}

/tmp/ipython-input-1042956962.py:115: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"merchant_id": merchant_id}, "observation": status, "ts": datetime.utcnow().isoformat()+"Z"
/tmp/ipython-input-1042956962.py:125: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"message": msg, "voucher": True}, "observation": note, "ts": datetime.utcnow().isoformat()+"Z"
/tmp/ipython-input-1042956962.py:132: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"driver_id": driver_id, "task": "short nearby delivery"},

...

GRABNAVI RESPONSE

Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.

GRABNAVI RESPONSE

Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.

{'combined_summary': 'Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.',
 'traces': [{'label': 'Overloaded Restaurant',
   'context': {'merchant_id': 'M123',
    'driver_id': 'DRV101',
    'cuisine': 'Food'},
   'steps': [{'step': 1,
     'incident': 'Overloaded Restaurant',
     'tool': 'get_merchant_status',
     'args': {'merchant_id': 'M123'},
     'observation': {'merchant_id': 'M123', 'prep_time': 40},
     'ts': '2025-09-09T19:31:53.166300Z'},
    {'step': 2,
     'incident': 'Overloaded Restaurant',
     'tool': 'notify_customer',
     'args': {'message': 'Your order from M123 has a longer prep time (~40 min). We’ve issued a small voucher for the delay.',
      'voucher': True},
     'observation': {'role': 'customer',
      'message': 'Your order from M123 has a longer prep time (~40 min). We’ve issued a small voucher for the delay.',
      'voucher_issued': True,
      'voucher_code': 'DELAY-5'},
     'ts': '2025-09-

In [ ]:
# ============================================================
# 🕒 Handle Customer Delay Complaint
# - Demonstrates how the system processes a complaint where
#   the customer reports late delivery or frustration.
# - This will route the input through the complaint handler
#   (handle_customer_delay_complaint) and return a structured response.
# ============================================================


await handle_text_case(
    "My food is not coming on time!! This is the worst."
)

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 0, 'distinct_labels': 1}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{
    'dispatch': 'single',
    'top_label': 'customer_complaint',
    'detected': [
        {
            'label': 'customer_complaint',
            'priority': 7,
            'confidence': 0.9,
            'reason': 'keyword',
            'matched_patterns': ['\\bworst\\b', '\\bnot coming on time\\b']
        }
    ]
}

GRABNAVI RESPONSE

{'note': "No runner mapped for label 'customer_complaint'. Falling back to compound resolver."}

Notify Customer: Customer complaint logged; sending apology + ETA.


GRABNAVI RESPONSE

fallback_compound

{'combined_summary': 'fallback_compound',
 'resolution': {'results': {'customer_complaint': {'status': 'logged'}},
  'log': [{'step': 'sorted_problems',
    'value': [{'label': 'customer_complaint',
      'priority': 7,
      'confidence': 0.9,
      'reason': 'keyword',
      'matched_patterns': ['\\bworst\\b', '\\bnot coming on time\\b']}]},
   {'step': 'invoking', 'label': 'customer_complaint', 'priority': 7},
   {'step': 'done',
    'label': 'customer_complaint',
    'result': {'status': 'logged'}}],
  'processed': ['customer_complaint']}}

In [ ]:
# ============================================================
# 🍕 Compound Case Demo: Overloaded Restaurant
# - Demonstrates the compound orchestrator handling a restaurant delay.
# - Scenario: Merchant PizzaPalace reports a 40-minute prep time,
#   and driver DRV101 is stuck waiting for the order.
# - The system should detect "Overloaded Restaurant" and respond
#   with actions like notifying customer, suggesting alternatives, etc.
# ============================================================

await handle_text_case(
    "merchant PizzaPalace showing 40-minute kitchen prep time, driver DRV101 waiting."
)

GRABNAVI RESPONSE

{
    'gate_decision': False,
    'meta': {'heuristic': {'keyword_count': 1, 'distinct_labels': 1}, 'final_source': 'heuristic_default'}
}

GRABNAVI RESPONSE

{
    'dispatch': 'single',
    'top_label': 'overloaded_restaurant',
    'detected': [
        {
            'label': 'overloaded_restaurant',
            'priority': 3,
            'confidence': 0.9,
            'reason': 'keyword',
            'matched_patterns': ['\\b(long prep|prep time|prep)\\b', '\\b40[- ]?minute\\b']
        }
    ]
}

/tmp/ipython-input-1042956962.py:115: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"merchant_id": merchant_id}, "observation": status, "ts": datetime.utcnow().isoformat()+"Z"
/tmp/ipython-input-1042956962.py:125: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"message": msg, "voucher": True}, "observation": note, "ts": datetime.utcnow().isoformat()+"Z"
/tmp/ipython-input-1042956962.py:132: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "args": {"driver_id": driver_id, "task": "short nearby delivery"},

...

GRABNAVI RESPONSE

Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.

GRABNAVI RESPONSE

Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.

{'combined_summary': 'Overloaded Restaurant: status checked, customer notified + voucher, driver rerouted, alternatives suggested.',
 'traces': [{'label': 'Overloaded Restaurant',
   'context': {'merchant_id': 'M123',
    'driver_id': 'DRV101',
    'cuisine': 'Food'},
   'steps': [{'step': 1,
     'incident': 'Overloaded Restaurant',
     'tool': 'get_merchant_status',
     'args': {'merchant_id': 'M123'},
     'observation': {'merchant_id': 'M123', 'prep_time': 40},
     'ts': '2025-09-09T19:31:53.629905Z'},
    {'step': 2,
     'incident': 'Overloaded Restaurant',
     'tool': 'notify_customer',
     'args': {'message': 'Your order from M123 has a longer prep time (~40 min). We’ve issued a small voucher for the delay.',
      'voucher': True},
     'observation': {'role': 'customer',
      'message': 'Your order from M123 has a longer prep time (~40 min). We’ve issued a small voucher for the delay.',
      'voucher_issued': True,
      'voucher_code': 'DELAY-5'},
     'ts': '2025-09-

# SendGrid Email Notification

In [ ]:
!pip install sendgrid

from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# 🔑 Replace with your SendGrid API key
SENDGRID_API_KEY = "YOUR_SENDGRID_API"

def send_driver_delay(to_email, driver_name="Rajesh", order_id="ORD1234", delay_time=15):
    """
    Send a professional Grab-styled delay notification email
    """
    html_content = f"""
    <div style="font-family: Arial, sans-serif; background:#f9f9f9; padding:20px;">
        <div style="max-width:600px; margin:auto; background:#fff; border-radius:10px;
                    border:1px solid #ddd; overflow:hidden;">

            <!-- Header -->
            <div style="background:#00B140; padding:15px; text-align:center; color:#fff;">
                <h1 style="margin:0;">Grab Delivery 🚦</h1>
            </div>

            <!-- Body -->
            <div style="padding:20px; color:#333;">
                <p style="font-size:16px;">Hello Customer,</p>

                <p style="font-size:15px;">
                    Your driver <b>{driver_name}</b> delivering order
                    <b style="color:#00B140;">{order_id}</b>
                    is currently stuck in traffic.
                </p>

                <div style="background:#fff8e1; border-left:5px solid #ffc107; padding:10px; margin:15px 0;">
                    <p style="margin:0; font-size:15px;">
                        ⏰ Expected Delay: <b>{delay_time} minutes</b>
                    </p>
                </div>

                <p style="font-size:14px; color:#555;">
                    We sincerely apologize for the inconvenience 🙏<br>
                    Thank you for choosing <b>Grab Delivery</b>. 🚀
                </p>
            </div>

            <!-- Footer -->
            <div style="background:#f1f1f1; padding:10px; text-align:center; font-size:12px; color:#777;">
                © 2025 Grab Delivery Hackathon | Prototype Notification
            </div>
        </div>
    </div>
    """

    message = Mail(
        from_email=("Grab Delivery 🚀 <thanmai.kamalapuram4@gmail.com>"),
        to_emails=to_email,
        subject=f"🚨 Delay Notification for Order {order_id}",
        html_content=html_content
    )

    try:
        sg = SendGridAPIClient(SENDGRID_API_KEY)
        response = sg.send(message)
        print(f"✅ Email sent successfully to {to_email} | Status: {response.status_code}")
    except Exception as e:
        print("❌ Error:", str(e))


# ------------------------
# 🚀 Demo Run (compile-time defined email)
# ------------------------
customer_email = "lasyapriya.275@gmail.com"   # 👈 define here directly
send_driver_delay(customer_email, driver_name="Rajesh", order_id="GRAB2025", delay_time=20)


Enter customer email: lasyapriya.275@gmail.com
✅ Email sent successfully to lasyapriya.275@gmail.com | Status: 202


# Route Calculation & Map Plotting

In [ ]:
# --- Step 1: Install dependencies ---
!pip install boto3 folium

import boto3
import folium

# --- Step 2: Configure AWS Credentials ---
aws_access_key = "USE YOUR AWS ACCESS KEY"
aws_secret_key = "USE YOUR AWS SECRET KEY"
aws_region = "USE YOUR AWS REGION"   # replace with your region

session = boto3.Session(
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name=aws_region
)

location = session.client("location")

# --- Step 3: Coordinates ---
start = [80.6234, 16.4420]   # SRM University AP (lon, lat)
end = [77.5946, 12.9716]     # Grab Office Bangalore (lon, lat)

# --- Step 4: Calculate Route ---
response = location.calculate_route(
    CalculatorName="GrabHackathonRoute",
    DeparturePosition=start,
    DestinationPosition=end
)

# --- Step 5: Print summary ---
print("✅ Distance (km):", response["Summary"]["Distance"])
print("✅ Duration (hours):", response["Summary"]["DurationSeconds"] / 3600)

# --- Step 6: Extract coordinates from Legs -> Steps ---
coords = []
for leg in response["Legs"]:
    for step in leg["Steps"]:
        coords.append((step["StartPosition"][1], step["StartPosition"][0]))  # (lat, lon)
    coords.append((leg["EndPosition"][1], leg["EndPosition"][0]))  # add final end point

# --- Step 7: Plot route ---
m = folium.Map(location=[15.0, 78.5], zoom_start=6)  # center between AP and Karnataka

folium.PolyLine(coords, color="blue", weight=3).add_to(m)

# Add markers
folium.Marker([16.4420, 80.6234], tooltip="SRM University AP", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([12.9716, 77.5946], tooltip="Grab Office Bangalore", icon=folium.Icon(color="red")).add_to(m)

m


✅ Distance (km): 652.829
✅ Duration (hours): 7.921388888888889


# DynamoDB

In [ ]:
# Cell A — Install and configure
!pip install flask boto3 gradio folium requests

# Python imports & AWS setup
import os, time, threading, json
from decimal import Decimal
from flask import Flask, request, jsonify
import boto3
import folium
import requests

# --- Replace these with your real values ---
AWS_ACCESS_KEY = "USE YOUR AWS ACCESS KEY"
AWS_SECRET_KEY = "USE YOUR AWS SECRET KEY"
AWS_REGION = "USE YOUR AWS REGION"                # change if needed
DDB_TABLE_NAME = "GrabUpdates"    # your DynamoDB table name (case sensitive)
ROUTER_NAME = "GrabHackathonRoute"      # your AWS Location Route Calculator name

# --- Create boto3 session & clients ---
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

dynamodb = session.resource("dynamodb")
table = dynamodb.Table(DDB_TABLE_NAME)
try:
    location = session.client("location")
except Exception as e:
    # If AWS Location isn't configured, calls will fail later but code will continue
    print("⚠️ Could not create AWS Location client:", e)

location = session.client("location")

# Flask

In [ ]:
# Cell B — Agent microservice (runs in background thread)
# This Flask app exposes POST /agent and returns JSON {ok, action, message, map_html, agent_raw}

from flask import Flask, request, jsonify
import traceback
import asyncio
import time

app = Flask("grabbot_agent")

# Hook into any existing agent in the notebook (if you implemented handle_text_case)
AGENT_SYNC = globals().get("sync_handle_text_case", None)
AGENT_ASYNC = globals().get("handle_text_case", None)

async def _stub_async(text, image_metadata=None, **kwargs):
    # fallback simple decision maker if your agent isn't available
    return {"combined_summary": text, "problems": [{"label": "other", "confidence": 0.5}], "note": "stub"}

# If no agent defined, use stub
if AGENT_SYNC is None and AGENT_ASYNC is None:
    AGENT_ASYNC = _stub_async

def call_agent_sync(text: str, image_metadata=None):
    """Synchronous wrapper to call your agent if present, otherwise fallback."""
    try:
        if AGENT_SYNC is not None:
            return AGENT_SYNC(text, image_metadata=image_metadata)
        else:
            # try to run async agent
            loop = None
            try:
                loop = asyncio.get_event_loop()
            except RuntimeError:
                loop = None
            if loop and loop.is_running():
                return asyncio.run(AGENT_ASYNC(text, image_metadata=image_metadata))
            else:
                return asyncio.get_event_loop().run_until_complete(AGENT_ASYNC(text, image_metadata=image_metadata))
    except Exception:
        # last-resort stub
        try:
            return asyncio.run(AGENT_ASYNC(text, image_metadata=image_metadata))
        except Exception:
            return {"error": "agent call failed", "trace": traceback.format_exc()}

# Simple decision function (same logic you had)
def decide_action(status_text: str, agent_resp: dict):
    s = (status_text or "").lower()
    if any(k in s for k in ("traffic", "stuck", "jam", "blocked", "delay on road")):
        return "traffic"
    if any(k in s for k in ("restaurant", "prep", "kitchen", "food delay")):
        return "restaurant_delay"
    if any(k in s for k in ("assault", "safety", "attack", "robbery", "injur")):
        return "safety"
    if any(k in s for k in ("tire", "flat", "vehicle", "breakdown")):
        return "vehicle"
    # fallback to agent problems
    try:
        probs = agent_resp.get("problems", []) if isinstance(agent_resp, dict) else []
        for p in probs:
            lbl = (p.get("label") if isinstance(p, dict) else str(p)).lower()
            if "traffic" in lbl or "jam" in lbl or "stuck" in lbl:
                return "traffic"
            if "restaurant" in lbl or "prep" in lbl or "overload" in lbl:
                return "restaurant_delay"
            if "safety" in lbl or "assault" in lbl:
                return "safety"
            if "vehicle" in lbl or "tire" in lbl:
                return "vehicle"
    except Exception:
        pass
    return "info"

# Helper: folium map HTML builder (from route response)
def make_map_html_from_route_response(route_response):
    try:
        coords = []
        for leg in route_response.get("Legs", []):
            for step in leg.get("Steps", []):
                sp = step.get("StartPosition")
                if sp and len(sp) >= 2:
                    coords.append((sp[1], sp[0]))  # lat, lon
            endp = leg.get("EndPosition")
            if endp and len(endp) >= 2:
                coords.append((endp[1], endp[0]))
        if not coords:
            return ""
        m = folium.Map(location=coords[0], zoom_start=12)
        folium.PolyLine(coords, weight=4).add_to(m)
        folium.Marker(coords[0], tooltip="Start").add_to(m)
        folium.Marker(coords[-1], tooltip="End").add_to(m)
        path = "agent_map.html"
        m.save(path)
        with open(path, "r", encoding="utf-8") as f:
            html = f.read()
        return f"<iframe srcdoc='{html}' width='100%' height='450'></iframe>"
    except Exception as e:
        return ""

# Flask route handler
@app.route("/agent", methods=["POST"])
def agent_handler():
    try:
        payload = request.get_json(force=True)
        order_id = payload.get("order_id")
        status = payload.get("status", "")
        lat = payload.get("lat", None)
        lon = payload.get("lon", None)

        # 1) call the agent
        agent_resp = call_agent_sync(status, image_metadata=None)

        # 2) decide action
        action = decide_action(status, agent_resp)

        # 3) prepare map + message
        map_html = ""
        user_message = ""
        route_summary = {}
        if action == "traffic":
            try:
                resp = location.calculate_route(
                    CalculatorName=ROUTER_NAME,
                    DeparturePosition=[80.6234, 16.4420],   # static example; replace if you have real coords
                    DestinationPosition=[77.5946, 12.9716]
                )
                map_html = make_map_html_from_route_response(resp)
                if "Summary" in resp:
                    route_summary["distance_km"] = resp["Summary"].get("Distance")
                    route_summary["duration_s"] = resp["Summary"].get("DurationSeconds")
                user_message = "🚦 Traffic detected — customer notified and route attached."
            except Exception as e:
                user_message = f"🚦 Traffic detected — but route generation failed: {str(e)}"
        elif action == "restaurant_delay":
            user_message = "🍽️ Restaurant delay — customer informed to wait."
        elif action == "safety":
            user_message = "⚠️ Safety incident — escalated to human support."
        elif action == "vehicle":
            user_message = "🛞 Vehicle issue — backup driver being arranged."
        else:
            user_message = f"✅ Logged: {status}"

        # 4) write log to DynamoDB — include agent_raw
        try:
            item = {
                "order_id": order_id,
                "timestamp": Decimal(str(int(time.time()))),
                "status": status
            }
            if lat is not None and lon is not None:
                item["lat"] = Decimal(str(lat))
                item["lon"] = Decimal(str(lon))
            if route_summary.get("distance_km") is not None:
                item["distance_km"] = Decimal(str(route_summary["distance_km"]))
            if route_summary.get("duration_s") is not None:
                item["duration_s"] = Decimal(str(route_summary["duration_s"]))

            # store agent response as JSON string for robustness
            try:
                item["agent_raw"] = json.dumps(agent_resp, default=str)
            except Exception:
                item["agent_raw"] = str(agent_resp)

            table.put_item(Item=item)
        except Exception as e:
            user_message += f"  (note: DB write failed: {str(e)})"

        # 5) return structured response
        return jsonify({
            "ok": True,
            "action": action,
            "message": user_message,
            "map_html": map_html,
            "agent_raw": agent_resp if isinstance(agent_resp, (dict, list, str)) else str(agent_resp)
        })

    except Exception as exc:
        return jsonify({"ok": False, "error": str(exc), "trace": traceback.format_exc()}), 500
# Root route for browser testing



# Start the Flask app in a background thread
def _run_agent_server():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=_run_agent_server, daemon=True)
t.start()
print("✅ Agent server started at http://127.0.0.1:5000/agent")

✅ Agent server started at http://127.0.0.1:5000/agent
 * Serving Flask app 'grabbot_agent'



# Driver Gradio

In [ ]:
# Cell C — Extended Driver UI with Logs & Trace (order-id defaults changed to GRAB20XX)
import gradio as gr
import requests
from pprint import pformat
from boto3.dynamodb.conditions import Key
import json
import time
import re
import boto3
from decimal import Decimal
from datetime import datetime

# --- SendGrid imports (from gradio code 2)
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# --- Config (use existing globals if present; otherwise create resource)
AGENT_URL = "http://127.0.0.1:5000/agent"
DRIVER_EVENT_URL = AGENT_URL

dynamodb = globals().get("dynamodb", None)
if dynamodb is None:
    try:
        session = boto3.Session()
        dynamodb = session.resource("dynamodb")
    except Exception:
        dynamodb = boto3.resource("dynamodb")

table = globals().get("table", None)
if table is None:
    try:
        table = dynamodb.Table("GrabUpdates")
    except Exception:
        table = None

orders_table = globals().get("orders_table", None)
if orders_table is None:
    try:
        orders_table = dynamodb.Table("Orders")
    except Exception:
        orders_table = None

# SendGrid config
SENDGRID_API_KEY = "USE YOUR SENDGRID API"
FROM_EMAIL = "Grab Delivery <thanmai.kamalapuram4@gmail.com>"

def send_customer_email(to_email, order_id, delay_time="≈15 minutes"):
    subject = f"🚨 Delay Notification for Order {order_id}"
    html_content = f"""
    <div style="font-family:Arial, sans-serif; background:#f9f9f9; padding:12px;">
        <div style="max-width:640px; margin:auto; background:#fff; border-radius:8px; border:1px solid #eee; overflow:hidden;">
            <div style="background:#00B140; padding:12px; color:white; text-align:center;">
                <h3 style="margin:0">Grab Delivery — Update</h3>
            </div>
            <div style="padding:16px; color:#333;">
                <p>Hello,</p>
                <p>Your driver for order <strong>{order_id}</strong> has an update:</p>
                <p><strong>Estimated delay:</strong> {delay_time}</p>
                <p>We’re sorry for the inconvenience — we’re working on it.</p>
                <p style="margin-top:14px; font-size:13px; color:#666;">Grab Delivery</p>
            </div>
        </div>
    </div>
    """
    try:
        message = Mail(
            from_email=FROM_EMAIL,
            to_emails=to_email,
            subject=subject,
            html_content=html_content
        )
        sg = SendGridAPIClient(SENDGRID_API_KEY)
        response = sg.send(message)
        return f"✅ Email sent to {to_email} | Status {getattr(response, 'status_code', 'unknown')}"
    except Exception as e:
        return f"❌ Email error: {e}"

import os, pandas as pd
EMAIL_LOG_FILE = globals().get("EMAIL_LOG_FILE", "email_logs.json")

def append_email_log(entry: dict):
    logs = []
    if os.path.exists(EMAIL_LOG_FILE):
        try:
            with open(EMAIL_LOG_FILE, "r", encoding="utf-8") as f:
                logs = json.load(f)
        except Exception:
            logs = []
    logs.append(entry)
    with open(EMAIL_LOG_FILE, "w", encoding="utf-8") as f:
        json.dump(logs, f, indent=2, ensure_ascii=False)

def load_email_logs_df():
    if not os.path.exists(EMAIL_LOG_FILE):
        return pd.DataFrame(columns=["timestamp", "to_email", "subject", "order_id", "driver", "delay_minutes", "status", "info"])
    try:
        with open(EMAIL_LOG_FILE, "r", encoding="utf-8") as f:
            logs = json.load(f)
    except Exception:
        logs = []
    df = pd.DataFrame(logs)
    cols = ["timestamp", "to_email", "subject", "order_id", "driver", "delay_minutes", "status", "info"]
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols]

def send_email_if_incident(status_text: str, order_id: str):
    if not order_id:
        return "No order_id provided; email not sent."
    if not status_text:
        return "No status text; email not sent."

    incident_keywords = ("traffic", "stuck", "jam", "restaurant", "prep", "assault", "safety", "tire", "flat", "vehicle", "delay")
    if not any(k in status_text.lower() for k in incident_keywords):
        return "Status is informational; no email sent."

    global orders_table
    if orders_table is None:
        try:
            orders_table = dynamodb.Table("Orders")
        except Exception:
            return "No Orders table handle available; cannot send email."

    try:
        resp = orders_table.get_item(Key={"order_id": order_id})
        item = resp.get("Item", {})
        customer_email = item.get("customer_email")
        driver_name = item.get("driver_name", "Driver")
        if not customer_email:
            return f"No customer_email found for order {order_id}."
    except Exception as e:
        return f"Orders lookup failed: {e}"

    delay_val = None
    try:
        m = re.search(r"(\d+)\s*(?:min|mins|minutes)", status_text, flags=re.IGNORECASE)
        if m:
            delay_val = int(m.group(1))
    except Exception:
        delay_val = None

    delay_text = f"{delay_val} minutes" if delay_val else "≈15 minutes"
    send_result = send_customer_email(customer_email, order_id, delay_text)

    log_entry = {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "to_email": customer_email,
        "subject": f"🚨 Delay Notification for Order {order_id}",
        "order_id": order_id,
        "driver": driver_name,
        "delay_minutes": delay_val if delay_val is not None else "",
        "status": "sent" if send_result.startswith("✅") else "send_error",
        "info": send_result
    }
    try:
        append_email_log(log_entry)
    except Exception:
        pass

    return send_result

def driver_update_via_agent(status, order_id):
    payload = {"order_id": order_id, "status": status}
    try:
        resp = requests.post(AGENT_URL, json=payload, timeout=20)
        if resp.status_code == 200:
            data = resp.json()
            if data.get("ok"):
                msg = data.get("message", "OK")
                map_html = data.get("map_html", "")
                trace = pformat(data.get("agent_raw", {}), indent=2, width=80)
                if isinstance(data.get("agent_raw"), str):
                    try:
                        trace = pformat(json.loads(data.get("agent_raw")), indent=2, width=80)
                    except Exception:
                        trace = data.get("agent_raw")

                try:
                    email_attempt = send_email_if_incident(status, order_id)
                    msg = f"{msg}  ({email_attempt})"
                except Exception as e:
                    msg = f"{msg}  (email attempt failed: {e})"

                return msg, map_html, trace
            else:
                return f"Agent error: {data.get('error')}", "", ""
        else:
            return f"Agent HTTP {resp.status_code}", "", ""
    except Exception as e:
        return f"❌ Agent call failed: {e}", "", ""

def fetch_full_trace(order_id_val):
    from boto3.dynamodb.conditions import Key as _Key
    try:
        resp = table.query(KeyConditionExpression=_Key("order_id").eq(order_id_val), ScanIndexForward=True)
        items = sorted(resp.get("Items", []), key=lambda x: int(x.get("timestamp", 0))) if resp else []
        if not items:
            return "No logs found for order_id: " + str(order_id_val)
        latest = items[-1]
        status_text = latest.get("status", "")
        if not status_text:
            return "Latest DB entry has no status."

        try:
            r = requests.post(DRIVER_EVENT_URL, json={"order_id": order_id_val, "status": status_text}, timeout=40)
            try:
                data = r.json()
            except Exception:
                return f"Agent returned non-JSON: {r.text}"
            for candidate in ("agent_raw", "trace", "result"):
                if isinstance(data, dict) and candidate in data:
                    val = data[candidate]
                    if isinstance(val, str):
                        try:
                            parsed = json.loads(val)
                            return pformat(parsed, indent=2, width=120)
                        except Exception:
                            return val
                    else:
                        return pformat(val, indent=2, width=120)
            return pformat(data, indent=2, width=120)
        except Exception as e:
            return f"❌ Error calling Flask agent: {e}"

    except Exception as e:
        return f"❌ Failed to fetch latest DB entry: {e}"

def fetch_logs(order_id):
    try:
        resp = table.query(
            KeyConditionExpression=Key("order_id").eq(order_id),
            ScanIndexForward=False,
            Limit=10
        )
        items = resp.get("Items", [])
        if not items:
            return "No logs found."
        lines = []
        for it in items:
            ts = int(it.get("timestamp", 0))
            status = it.get("status", "")
            dist = it.get("distance_km")
            dur = it.get("duration_s")
            line = f"🕒 {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(ts))} — {status}"
            if dist and dur:
                line += f" (Route {dist} km, {dur} s)"
            lines.append(line)
        return "\n".join(lines)
    except Exception as e:
        return f"❌ Failed to fetch logs: {e}"

def send_driver_delay_and_log(to_email, driver_name="Rajesh", order_id="ORD1234", delay_time=15, actually_send=False):
    timestamp = datetime.utcnow().isoformat() + "Z"
    subject = f"🚨 Delay Notification for Order {order_id}"

    log_entry = {
        "timestamp": timestamp,
        "to_email": to_email,
        "subject": subject,
        "order_id": order_id,
        "driver": driver_name,
        "delay_minutes": delay_time,
    }

    status_message = ""
    if actually_send:
        try:
            send_res = send_customer_email(to_email, order_id, f"{delay_time} minutes")
            if send_res.startswith("✅"):
                log_entry["status"] = "sent"
                status_message = f"Email sent to {to_email}"
            else:
                log_entry["status"] = "send_error"
                log_entry["info"] = send_res
                status_message = f"Error sending email: {send_res}"
        except Exception as e:
            log_entry["status"] = "send_error"
            log_entry["info"] = str(e)
            status_message = f"Error sending email: {e}"
    else:
        log_entry["status"] = "simulated"
        status_message = f"Email simulated for {to_email}"

    append_email_log(log_entry)
    return load_email_logs_df(), status_message

# -----------------------
# Build Gradio UI (kept unchanged layout) but order-id defaults use GRAB20XX
# -----------------------
with gr.Blocks(css="""
    body { background-color: #ffffff; font-family: 'Segoe UI', sans-serif; }
    .grab-btn {
        background: linear-gradient(
            135deg,
            rgba(57, 166, 90, 0.85),
            rgba(47, 138, 75, 0.75)
        ) !important;
        color: #fff !important;
        font-weight: 600;
        font-size: 15px;
        border-radius: 10px;
        padding: 12px 22px !important;
        border: 1px solid rgba(57, 166, 90, 0.5) !important;
        box-shadow: 0px 3px 8px rgba(0,0,0,0.08);
        backdrop-filter: blur(6px);
        transition: all 0.25s ease;
    }
    .grab-btn:hover {
        background: linear-gradient(
            135deg,
            rgba(47, 138, 75, 0.9),
            rgba(57, 166, 90, 0.85)
        ) !important;
        transform: translateY(-2px);
        box-shadow: 0px 6px 14px rgba(0,0,0,0.15);
    }
    .grab-btn:active {
      transform: translateY(0px);
      box-shadow: 0px 2px 6px rgba(0,0,0,0.1);
    }
    .grab-card {
        border: 1px solid #39a65a;
        border-radius: 14px;
        padding: 18px;
        background-color: #f9fff9;
        margin-bottom: 14px;
        box-shadow: 0px 2px 6px rgba(0,0,0,0.05);
    }
    .grab-banner img {
        display: block;
        margin: 0 auto;
        width: 100%;
        max-width: 1600px;
        border-radius: 12px;
    }
    .grab-quick-row button {
        flex: 1;
        margin: 4px;
    }
""") as driver_ui:

    with gr.Row():
        with gr.Column():
            gr.Image("grabnavi-banner.png", show_label=False, elem_classes="grab-banner")

    with gr.Tab("➕ New Update"):
        with gr.Row():
            with gr.Column(scale=1, min_width=350):
                with gr.Group(elem_classes="grab-card"):
                    # DEFAULT CHANGED: user-visible variant GRAB20XX (no fixed GRAB2025)
                    order_id = gr.Textbox(label="Order ID", value="GRAB20XX", placeholder="e.g. GRAB2012")
                    status = gr.Textbox(label="Update Status", placeholder="Type or choose quick action...")
                    submit_btn = gr.Button("Submit Update", elem_classes="grab-btn", scale=1)

                    gr.Markdown("### Quick Actions")
                    with gr.Row():
                        btn_traffic = gr.Button("🚦 Stuck in Traffic", elem_classes="grab-btn")
                        btn_restaurant = gr.Button("🍽️ Restaurant Delay", elem_classes="grab-btn")
                    with gr.Row():
                        btn_safety = gr.Button("⚠️ Safety Incident", elem_classes="grab-btn")
                        btn_vehicle = gr.Button("🛞 Flat Tire", elem_classes="grab-btn")

            with gr.Column(scale=2):
                with gr.Group(elem_classes="grab-card"):
                    output = gr.Textbox(label="System Response", interactive=False)
                with gr.Group(elem_classes="grab-card"):
                    map_display = gr.HTML(label="Preview (map if any)")
                with gr.Group(elem_classes="grab-card"):
                    trace_display = gr.Textbox(label="Agent Trace", lines=12, interactive=False)

        submit_btn.click(driver_update_via_agent, [status, order_id], [output, map_display, trace_display])
        btn_traffic.click(lambda oid: driver_update_via_agent("Traffic jam at Airport Road — driver stuck in jam", oid), [order_id], [output, map_display, trace_display])
        btn_restaurant.click(lambda oid: driver_update_via_agent("Restaurant delay: 40 min — PizzaPalace showing 40-minute kitchen prep time; driver DRV101 waiting", oid), [order_id], [output, map_display, trace_display])
        btn_safety.click(lambda oid: driver_update_via_agent("Safety incident: driver assaulted, needs urgent help", oid), [order_id], [output, map_display, trace_display])
        btn_vehicle.click(lambda oid: driver_update_via_agent("Flat tire: driver DRV101 has flat tire; package may be at risk", oid), [order_id], [output, map_display, trace_display])

    with gr.Tab("🧾 View Full Trace"):
        with gr.Group(elem_classes="grab-card"):
            trace_order_id = gr.Textbox(label="Order ID (for trace)", value="GRAB20XX", placeholder="e.g. GRAB2012")
            fetch_trace_btn = gr.Button("Fetch Full Trace", elem_classes="grab-btn")
            full_trace_box = gr.Textbox(label="Full Agent Trace (pretty)", lines=25, interactive=False)

        fetch_trace_btn.click(fetch_full_trace, [trace_order_id], [full_trace_box])

    with gr.Tab("📜 View Logs"):
        with gr.Group(elem_classes="grab-card"):
            order_id_logs = gr.Textbox(label="Order ID", value="GRAB20XX", placeholder="e.g. GRAB2012")
            fetch_btn = gr.Button("Fetch Logs", elem_classes="grab-btn")
            logs_box = gr.Textbox(label="Recent Logs", lines=15, interactive=False)

        fetch_btn.click(fetch_logs, [order_id_logs], [logs_box])

    with gr.Tab("📧 Email Logs"):
        with gr.Group(elem_classes="grab-card"):
            email_to = gr.Textbox(label="Customer Email", value="")
            # DEFAULT CHANGED: email Order id textbox to GRAB20XX (user must type actual id)
            email_order = gr.Textbox(label="Order ID", value="GRAB20XX", placeholder="e.g. GRAB2012")
            email_driver = gr.Textbox(label="Driver name", value="Rajesh")
            email_delay = gr.Number(label="Delay (minutes)", value=20)
            simulate_checkbox = gr.Checkbox(label="Simulate (don't actually send)", value=True)
            send_email_btn = gr.Button("Send Email", elem_classes="grab-btn")
            email_status = gr.Textbox(label="Status", interactive=False)

            gr.Markdown("### Email logs (persistent)")
            logs_df = gr.Dataframe(value=load_email_logs_df(), label="Email Logs", interactive=False)
            refresh_logs_btn = gr.Button("Refresh logs", elem_classes="grab-btn")

        def _on_send_email(to_email, order_id_val, driver_name_val, delay_val, simulate):
            df, status = send_driver_delay_and_log(
                to_email=to_email,
                driver_name=driver_name_val or "Rajesh",
                order_id=order_id_val or "ORD1234",
                delay_time=int(delay_val or 0),
                actually_send=(not simulate)
            )
            return df, status

        send_email_btn.click(
            fn=_on_send_email,
            inputs=[email_to, email_order, email_driver, email_delay, simulate_checkbox],
            outputs=[logs_df, email_status]
        )

        refresh_logs_btn.click(fn=lambda: load_email_logs_df(), inputs=[], outputs=[logs_df])

driver_ui.launch(share=True)


# Customer Gradio

In [ ]:
# gradio_app_final.py

!pip install boto3 gradio
import requests

import gradio as gr
import boto3
from botocore.exceptions import ClientError
from boto3.dynamodb.conditions import Key
import uuid
import datetime
import re



# ---------------------------
# AWS / DynamoDB Configuration
# ---------------------------
aws_access_key = "USE YOUR AWS ACCESS KEY"
aws_secret_key = "USE YOUR AWS SECRET KEY"
aws_region = "USE YOUR AWS REGION"

dynamodb = boto3.resource(
    "dynamodb",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name=aws_region
)

# Table objects (make sure these tables exist in your AWS account)
claims_table_db = dynamodb.Table("claims")
notifications_table_db = dynamodb.Table("notifications")
customers_table_db = dynamodb.Table("customers")
orders_table_db = dynamodb.Table("Orders")

# ---------------------------
# Helpers
# ---------------------------
def utc_now_readable():
    return datetime.datetime.utcnow().strftime("%A, %d %B %Y %I:%M %p")

def utc_iso():
    return datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

def is_valid_email(email):
    return re.match(r"[^@]+@[^@]+\.[^@]+", email or "") is not None

# Safe wrappers for DB reads so UI doesn't crash on errors
def safe_query_table(table, key_name, key_value):
    try:
        resp = table.query(KeyConditionExpression=Key(key_name).eq(key_value))
        return resp.get("Items", [])
    except Exception as e:
        print(f"DB query error for {table.name}: {e}")
        return []

# ---------------------------
# Core DB Functions
# ---------------------------
def save_customer_email(email):
    try:
        customers_table_db.put_item(Item={"email": email, "created": utc_iso()})
        return True
    except Exception as e:
        print("Error saving customer:", e)
        return False

def save_order_to_db(order_id, customer_email):
    if not order_id or not customer_email:
        return "⚠️ Please provide both order_id and customer_email"
    try:
        orders_table_db.put_item(Item={"order_id": order_id, "customer_email": customer_email})
        return f"✅ Order {order_id} saved for {customer_email}"
    except Exception as e:
        print("Error saving order:", e)
        return f"❌ DynamoDB Error: {e}"

def file_claim_db(email, order_id, description, image):
    claim_id = f"C{uuid.uuid4().hex[:8].upper()}"
    ts = utc_now_readable()
    iso_ts = utc_iso()
    item = {
        "email": email,
        "claim_id": claim_id,
        "order_id": order_id or "",
        "type": "Damage Parcel",
        "status": "New",
        "created": ts,
        "last_update": ts,
        "resolution": "Pending review",
        "description": description or "",
        "image": image or "",
        "time": iso_ts              # <-- NEW FIELD
    }
    try:
        claims_table_db.put_item(Item=item)
        # Also add a notification entry
        notifications_table_db.put_item(Item={
            "email": email,
            "notif_id": str(uuid.uuid4()),
            "type": "notification",
            "content": f"📢 Claim {claim_id} filed for Order {order_id}",
            "message": "Claim created",
            "timestamp": iso_ts
        })
        return f"✅ Claim {claim_id} filed\n\n Damage Detected! No need to worry, It will be evaluated and refund will be issued."
    except Exception as e:
        print("Error filing claim:", e)
        return f"❌ Error filing claim: {e}"


def get_claims_for_user(email):
    return safe_query_table(claims_table_db, "email", email)

def get_notifications_for_user(email):
    return safe_query_table(notifications_table_db, "email", email)

def get_messages_for_user(email):
    items = safe_query_table(notifications_table_db, "email", email)
    # messages are stored as type == "message"
    msgs = [it for it in items if it.get("type") == "message"]
    msgs.sort(key=lambda x: x.get("timestamp", ""))
    return msgs

def send_message_db(email, content, role="user"):
    try:
        notifications_table_db.put_item(Item={
            "email": email,
            "notif_id": str(uuid.uuid4()),
            "type": "message",
            "content": content,
            "message": "User message",
            "timestamp": utc_iso()
        })
        return "✅ Message saved"
    except Exception as e:
        print("Error saving message:", e)
        return f"❌ Error saving message: {e}"

# ---------------------------
# Gradio UI (only UI changed)
# ---------------------------
with gr.Blocks(css="""
    body { background: #fffdf9 !important; font-family: 'Segoe UI', Roboto, Arial; margin: 0; }
    .login-box { background:white; padding:28px; border-radius:12px;
        box-shadow:0 6px 18px rgba(0,0,0,0.06); max-width:420px; margin:60px auto; text-align:center; }
    /* Green menu background, buttons styled green */
    .menu { background: linear-gradient(180deg,#0aad5c,#0a8a47); padding: 18px; color: white; min-height: 100vh; width: 220px; }

    .menu .gr-button { width:100%; margin-bottom:10px; background: linear-gradient(180deg,#0aad5c,#0a8a47) !important; color:white !important; border:none !important; border-radius:8px; padding:10px 12px; }
    .menu .gr-button:hover { background:#0a8a47 !important; }
    /* All action buttons green */
    .gr-button { background:green !important; color:white !important; border:none !important; border-radius:8px; padding:10px 16px; }
    .gr-button:hover { background:#0a8a47 !important; }
    .content { margin-left: 260px; padding: 24px; }
    .card { border: 1px solid #e6f3ea; border-radius:10px; padding:16px; background:#f9fff9; margin-bottom:12px; }
    /* Make header area slightly larger on dashboard */
    .dashboard-header { padding: 12px 16px; border-radius:10px; background: linear-gradient(180deg,#f5fff5,#ecfff0); border:1px solid #e6f3ea; }
   .dashboard-view  {
    margin-left: -120px;   /* shift dashboard closer to menu */
    max-width: 900px;      /* optional: keep width clean */
}

    /* Sign In button inside login box */
.login-box .login-btn {
    background: linear-gradient(180deg,#0aad5c,#0a8a47) !important;
    color: white !important;
    font-weight: bold;
    border-radius: 8px;
    padding: 10px 16px;
}

.login-box .login-btn:hover {
    background: linear-gradient(180deg,#089044,#0a6e38) !important;
}
/* Menu buttons green gradient like Sign In */
.menu-btn {
    width:100% !important;
    margin-bottom:10px;
    background: linear-gradient(180deg,#0aad5c,#0a8a47) !important;
    color:white !important;
    border:none !important;
    border-radius:8px;
    padding:10px 12px;
    font-weight: bold;
}

.menu-btn:hover {
    background: linear-gradient(180deg,#089044,#0a6e38) !important;
}

/* Login box styling like Mangools */
.login-box {
    background: white;
    padding: 40px 32px;
    border-radius: 16px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.08);
    max-width: 420px;
    margin: 80px auto;
    text-align: center;
}

#login-logo {
    margin-bottom: 12px;
}

.login-box h2, .login-box h3, .login-box h1 {
    margin-bottom: 20px;
    font-weight: 600;
    color: #222;
}

/* Inputs styled like Mangools */
.login-box input {
    border: 1px solid #ddd !important;
    border-radius: 8px !important;
    padding: 10px 14px !important;
    font-size: 15px !important;
    margin-bottom: 14px !important;
}

/* Sign in button */
.login-box .login-btn {
    background: linear-gradient(180deg,#0aad5c,#0a8a47) !important;
    color: white !important;
    font-weight: bold !important;
    font-size: 16px !important;
    border-radius: 10px !important;
    padding: 12px !important;
    margin-top: 10px;
    width: 100% !important;
}

.login-box .login-btn:hover {
    background: linear-gradient(180deg,#089044,#0a6e38) !important;
}

""") as demo:

    session = gr.State({"email": None})

    # ---- LOGIN PAGE ----
    with gr.Column(visible=True) as login_page:
        with gr.Column(elem_classes="login-box"):
            gr.Image("/content/Grablogo.png", elem_id="login-logo", show_label=False, scale=0, height=60)
            gr.Markdown("## 👋 Good to see you again")
            login_order_id = gr.Textbox(label="Enter Your Order ID", placeholder="Enter Order ID (optional)")
            login_email = gr.Textbox(label="Enter Your Email", placeholder="you@example.com")
            btn_login = gr.Button("Sign in", elem_classes="login-btn")
            login_status = gr.Markdown("")

    # ---- APP LAYOUT (hidden until login) ----
    with gr.Row(visible=False) as app_layout:
        # Sidebar / Menu
        with gr.Column(elem_classes="menu", scale=1, min_width=220):
            btn_dashboard = gr.Button("🏠 Dashboard", elem_classes="menu-btn")
            btn_file_claim = gr.Button("📦 File Claim", elem_classes="menu-btn")
            btn_my_claims = gr.Button("📊 My Claims", elem_classes="menu-btn")
            btn_notifications = gr.Button("🔔 Notifications", elem_classes="menu-btn")
            btn_messages = gr.Button("💬 Messages", elem_classes="menu-btn")
            btn_save_order = gr.Button("🛒 Save Order", elem_classes="menu-btn")
            btn_logout = gr.Button("🚪 Logout", elem_classes="menu-btn")

        # Content area (multiple pages inside, toggle visibility)
        with gr.Column(scale=4, elem_classes="content"):

            # Dashboard view (contains welcome header — only place welcome is shown)
            with gr.Column(visible=True,elem_classes="dashboard-view") as dashboard_view:
                header_md = gr.Markdown("", elem_classes="dashboard-header")


                # dashboard card for other dynamic content
                dashboard_card = gr.Markdown("", elem_id="dashboard_card")

            # File Claim view
            with gr.Column(visible=False) as file_claim_view:
                with gr.Row():
                    gr.Markdown("## 📦 File Damage Parcel Claim")
                claim_order_id = gr.Textbox(label="Order ID")
                claim_desc = gr.Textbox(label="Describe the issue", lines=3)
                claim_img = gr.Image(type="filepath", label="Upload Parcel Image (optional)")
                claim_submit = gr.Button("Submit Claim", elem_classes="menu-btn")
                claim_out = gr.Markdown()

            # My Claims view
            with gr.Column(visible=False) as myclaims_view:
                gr.Markdown("## 📊 My Claims")
                claim_filter = gr.Dropdown(["All", "New", "In Progress", "Closed"], value="All", label="Filter by Status")
                btn_refresh_claims = gr.Button("Refresh Claims", elem_classes="menu-btn")
                claims_table = gr.Dataframe(
                    headers=["Claim ID", "Type", "Order ID", "Status", "Created", "Last Update", "Resolution"],
                    interactive=False
                )

            # Notifications view
            with gr.Column(visible=False) as notif_view:
                gr.Markdown("## 🔔 Notifications")
                notif_box = gr.Markdown()

            # Messages view
            with gr.Column(visible=False) as messages_view:
                gr.Markdown("## 💬 Messages with Support")
                msgs_md = gr.Markdown()
                msg_box = gr.Textbox(label="Type your message", placeholder="Hi, I need help with order ...")
                btn_send_msg = gr.Button("Send", elem_classes="menu-btn")
                msg_out = gr.Markdown()

            # Save Order view
            with gr.Column(visible=False) as saveorder_view:
                gr.Markdown("## 🛒 Save Order")
                save_order_id = gr.Textbox(label="Order ID")
                save_order_email = gr.Textbox(label="Customer Email")
                btn_save_order_submit = gr.Button("Save Order", elem_classes="menu-btn")
                save_order_out = gr.Markdown()

    # ---------------------------
    # Helper UI functions
    # ---------------------------
    def build_notifications_html(email):
        items = get_notifications_for_user(email)
        if not items:
            return "<span style='color:black'>✅ No new notifications</span>"

        html_list = []
        for it in items:
            content = it.get("content") or "📭 (no content)"
            ts = it.get("timestamp") or ""
            html_list.append(f"- {content} {f'({ts})' if ts else ''}")
        return "<br/>".join(html_list)


    def build_dashboard_html(email):
        # Welcome header only appears here (dashboard)
        username = email.split("@")[0] if email else "User"
    # Remove numbers and symbols, keep only alphabets
        username = re.sub(r'[^a-zA-Z]', '', username)
    # Capitalize first letter, lowercase rest
        username = username.capitalize() if username else "User"
        html = f"### Welcome to Your Grab Support Hub!\n\n 👋 Hey , **{username}**! You can file claims, track them, check notifications, and chat with support.\n\n"

        notes = get_notifications_for_user(email)[:5]
        if notes:
            html += "#### Recent Notifications\n"
            for n in notes:
                content = n.get("content") or "📭 (no content)"
                ts = n.get("timestamp") or ""
                html += f"- {content} {f'({ts})' if ts else ''}\n"
        else:
            html += "✅ No recent notifications\n"
        return html


    # ---------------------------
    # Login function (now includes order_id)
    # outputs required: login_status, session, login_page visibility, app_layout visibility, header_md, notif_dash
    def login_fn(email, order_id_in, session):
        if not email or not is_valid_email(email):
            return "⚠️ Please enter a valid email address.", session, gr.update(visible=True), gr.update(visible=False), "", ""
        # Save customer and order
        save_customer_email(email)
        save_msg = ""
        if order_id_in:
            save_msg = save_order_to_db(order_id_in, email)
        # Save session
        session["email"] = email
        # Prepare header and notifications (dashboard only)
        header = build_dashboard_html(email)
        notif_html = build_notifications_html(email)
        status = f"✅ Logged in as {email}. {save_msg}"
        return status, session, gr.update(visible=False), gr.update(visible=True), header, notif_html

    btn_login.click(
        fn=login_fn,
        inputs=[login_email, login_order_id, session],
        outputs=[login_status, session, login_page, app_layout, header_md, notif_box]
    )

    # ---------------------------
    # Page switching helper
    # ---------------------------
    def show_page(page_name):
        return (
            gr.update(visible=(page_name == "dashboard")),
            gr.update(visible=(page_name == "file_claim")),
            gr.update(visible=(page_name == "my_claims")),
            gr.update(visible=(page_name == "notifications")),
            gr.update(visible=(page_name == "messages")),
            gr.update(visible=(page_name == "save_order"))
        )

    btn_dashboard.click(lambda: show_page("dashboard"), None, [dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view])
    btn_file_claim.click(lambda: show_page("file_claim"), None, [dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view])
    btn_my_claims.click(lambda: show_page("my_claims"), None, [dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view])
    btn_notifications.click(lambda: show_page("notifications"), None, [dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view])
    btn_messages.click(lambda: show_page("messages"), None, [dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view])
    # show save order and prefill email
    def show_save_and_prefill(session):
        updates = show_page("save_order")
        email = session.get("email", "")
        return (*updates, gr.update(value=email))
    btn_save_order.click(fn=show_save_and_prefill, inputs=[session], outputs=[dashboard_view, file_claim_view, myclaims_view, notif_view, messages_view, saveorder_view, save_order_email])

    # ---------------------------
    # Dashboard click: refresh header and notif dash
    def refresh_dashboard(session):
        email = session.get("email", "")
        return build_dashboard_html(email), build_notifications_html(email)
    btn_dashboard.click(fn=refresh_dashboard, inputs=[session], outputs=[header_md, notif_box])

    # ---------------------------
    # Submit claim handler
    def handle_submit_claim(order_id_val, desc_val, image_val, session):
        email = session.get("email", "")
        if not email:
            return "⚠️ Please login first."
        result = file_claim_db(email, order_id_val, desc_val, image_val if image_val else "")
        return result
    claim_submit.click(fn=handle_submit_claim, inputs=[claim_order_id, claim_desc, claim_img, session], outputs=[claim_out])

    # ---------------------------
    # Refresh claims table
    def refresh_claims(filter_status_val, session):
        email = session.get("email", "")
        if not email:
            return []
        items = get_claims_for_user(email)
        if filter_status_val and filter_status_val != "All":
            items = [it for it in items if it.get("status") == filter_status_val]
        rows = []
        for c in items:
            rows.append([c.get("claim_id",""), c.get("type",""), c.get("order_id",""), c.get("status",""), c.get("created",""), c.get("last_update",""), c.get("resolution","")])
        return rows
    btn_refresh_claims.click(fn=refresh_claims, inputs=[claim_filter, session], outputs=[claims_table])

    # ---------------------------
    # Notifications view refresh
    def refresh_notifications(session):
        email = session.get("email", "")
        return build_notifications_html(email)
    btn_notifications.click(fn=refresh_notifications, inputs=[session], outputs=[notif_box])

    # ---------------------------
    # Messages: send and refresh
    def send_message_handler(msg_text, session):
        email = session.get("email", "")
        if not email:
            return "⚠️ Please login first."
        res = send_message_db(email, msg_text, role="user")
        # return updated messages markdown and empty input
        msgs = get_messages_for_user(email)
        if not msgs:
            msgs_md = "No messages yet."
        else:
            msgs_md = "\n\n".join([f"[{m.get('origin','user')}] {m.get('content')} ({m.get('timestamp')})" for m in msgs])
        return msgs_md, ""
    btn_send_msg.click(fn=send_message_handler, inputs=[msg_box, session], outputs=[msgs_md, msg_box])

    # When user navigates to messages, show current messages
    def show_messages_page(session):
        email = session.get("email", "")
        msgs = get_messages_for_user(email)
        if not msgs:
            return "No messages yet."
        return "\n\n".join([f"[{m.get('origin','user')}] {m.get('content')} ({m.get('timestamp')})" for m in msgs])
    btn_messages.click(fn=show_messages_page, inputs=[session], outputs=[msgs_md])

    # ---------------------------
    # Save Order from UI
    def save_order_handler(order_id_val, email_val, session):
        if not order_id_val or not email_val:
            return "⚠️ Please provide both Order ID and Email"
        res = save_order_to_db(order_id_val, email_val)
        return res
    btn_save_order_submit.click(fn=save_order_handler, inputs=[save_order_id, save_order_email, session], outputs=[save_order_out])

    # ---------------------------
    # Logout
    def logout_fn():
        # hide app and show login; clear session
        return {"email": None}, gr.update(visible=True), gr.update(visible=False), "### Logged out"
    btn_logout.click(fn=logout_fn, outputs=[session, login_page, app_layout, header_md])

# ---------------------------
# Launch
# ---------------------------
demo.launch(server_name="0.0.0.0", server_port=8097)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5cdb8ab7ca30325e3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
